In [5]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, RandomizedSearchCV, cross_val_score, KFold
from sklearn.preprocessing import StandardScaler, RobustScaler, QuantileTransformer, MinMaxScaler, MaxAbsScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.ensemble import AdaBoostRegressor
from sklearn.tree import DecisionTreeRegressor
import difflib
import warnings
import time
import os
from scipy import stats
warnings.filterwarnings('ignore')

# Advanced metrics
def relative_absolute_error(y_true, y_pred):
    return np.sum(np.abs(y_true - y_pred)) / np.sum(np.abs(y_true - np.mean(y_true)))

def root_relative_squared_error(y_true, y_pred):
    return np.sqrt(np.sum((y_true - y_pred) ** 2) / np.sum((y_true - np.mean(y_true)) ** 2))

def willmotts_index_of_agreement(y_true, y_pred):
    return 1 - np.sum((y_true - y_pred) ** 2) / np.sum((np.abs(y_true - np.mean(y_true)) + np.abs(y_pred - np.mean(y_true))) ** 2)

def nash_sutcliffe_efficiency(y_true, y_pred):
    return 1 - np.sum((y_true - y_pred) ** 2) / np.sum((y_true - np.mean(y_true)) ** 2)

def mean_absolute_percentage_error(y_true, y_pred):
    return np.mean(np.abs((y_true - y_pred) / y_true)) * 100

def explained_variance_score(y_true, y_pred):
    return 1 - np.var(y_true - y_pred) / np.var(y_true)

def median_absolute_error(y_true, y_pred):
    return np.median(np.abs(y_true - y_pred))

def max_error(y_true, y_pred):
    return np.max(np.abs(y_true - y_pred))

# Create results directory
results_dir = r"D:\2026 Work\My Papers\Predicting calcium carbonate yield\Models\AdaBoost"
os.makedirs(results_dir, exist_ok=True)

# Load your data
print("="*80)
print("LOADING DATA")
print("="*80)
df = pd.read_csv(r'D:\2026 Work\My Papers\Predicting calcium carbonate yield\Data\Data.csv', encoding='ISO-8859-1')
print("Columns:", df.columns.tolist())
df.columns = df.columns.str.strip()  
print("Cleaned columns:", df.columns.tolist())

# Find target column
target_col = 'calcium carbonate yield (%)'
if target_col not in df.columns:
    matches = difflib.get_close_matches('calcium carbonate yield', df.columns, n=1)
    if matches:
        target_col = matches[0]
        print(f"Using '{target_col}' as target column")
    else:
        print(f"ERROR: Could not find 'calcium carbonate yield (%)' or similar in columns!")
        print("Available columns:", df.columns.tolist())
        exit()

X = df.drop(columns=[target_col])
y = df[target_col]

# Advanced data preprocessing
print("\n" + "="*80)
print("ADVANCED DATA PREPROCESSING")
print("="*80)

# 1. Handle non-numeric columns
print("\n1. Handling non-numeric columns...")
non_numeric_columns = X.select_dtypes(exclude=[np.number]).columns.tolist()
if non_numeric_columns:
    print(f"Converting non-numeric columns: {non_numeric_columns}")
    for col in non_numeric_columns:
        X[col] = pd.to_numeric(X[col], errors='coerce')

# 2. Handle missing values
print("\n2. Handling missing values...")
print(f"Missing values before: {X.isna().sum().sum()}")
X = X.fillna(X.median())
print(f"Missing values after: {X.isna().sum().sum()}")

# 3. Remove outliers using IQR method
print("\n3. Removing outliers...")
initial_rows = len(X)
z_scores = np.abs(stats.zscore(X.select_dtypes(include=[np.number])))
X = X[(z_scores < 3).all(axis=1)]
y = y[X.index]
print(f"Removed {initial_rows - len(X)} outlier rows")

# 4. Feature engineering - create interaction terms (only if not too many features)
print("\n4. Creating interaction terms...")
if X.shape[1] < 20:
    from sklearn.preprocessing import PolynomialFeatures
    poly = PolynomialFeatures(degree=2, interaction_only=True, include_bias=False)
    X_poly = poly.fit_transform(X)
    feature_names = poly.get_feature_names_out(X.columns)
    X = pd.DataFrame(X_poly, columns=feature_names)
    print(f"Features expanded from {len(X.columns)} to {X.shape[1]}")
else:
    print(f"Keeping original {X.shape[1]} features")

# 5. Check target distribution
print("\n5. Target variable analysis...")
print(f"Target statistics - Mean: {y.mean():.2f}, Std: {y.std():.2f}")
print(f"Skewness: {y.skew():.4f}")
print(f"Kurtosis: {y.kurtosis():.4f}")

# 6. Stratified split
print("\n6. Creating stratified split...")
try:
    y_bins = pd.qcut(y, q=5, labels=False, duplicates='drop')
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y_bins
    )
except:
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"\nData shapes - Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Train target stats - Mean: {y_train.mean():.2f}, Std: {y_train.std():.2f}")
print(f"Test target stats - Mean: {y_test.mean():.2f}, Std: {y_test.std():.2f}")

# ====================================================================
# OPTIMIZED ADABOOST WITH PROPER PARAMETER SAMPLING
# ====================================================================
print("\n" + "="*80)
print("OPTIMIZED ADABOOST REGRESSOR")
print("="*80)

# Try different scalers
scalers = {
    'StandardScaler': StandardScaler(),
    'RobustScaler': RobustScaler(),
    'QuantileTransformer': QuantileTransformer(output_distribution='normal', n_quantiles=1000),
    'MinMaxScaler': MinMaxScaler(),
    'MaxAbsScaler': MaxAbsScaler()
}

best_score = np.inf
best_scaler = None
best_params = None
best_model = None
best_scaler_name = None

# Use DecisionTree as base estimator for AdaBoost (default)
base_estimator = DecisionTreeRegressor(max_depth=3, random_state=42)

for scaler_name, scaler in scalers.items():
    print(f"\n{'-'*60}")
    print(f"TRAINING WITH {scaler_name.upper()}")
    print(f"{'-'*60}")
    
    # Create pipeline with AdaBoost
    ada_pipeline = Pipeline([
        ('scaler', scaler),
        ('ada', AdaBoostRegressor(
            estimator=base_estimator,
            random_state=42,
        ))
    ])
    
    # Optimized parameter space for AdaBoost
    param_dist = {
        'ada__n_estimators': [50, 100, 150, 200, 250, 300, 400, 500],
        'ada__learning_rate': [0.01, 0.05, 0.1, 0.5, 1.0, 1.5, 2.0],
        'ada__loss': ['linear', 'square', 'exponential'],
        'ada__estimator__max_depth': [1, 2, 3, 4, 5],
        'ada__estimator__min_samples_split': [2, 5, 10],
        'ada__estimator__min_samples_leaf': [1, 2, 4],
        'ada__estimator__max_features': [None, 'sqrt', 'log2'],
    }
    
    total_combinations = 1
    for key in param_dist:
        total_combinations *= len(param_dist[key])
    
    print(f"Parameter space size: ~{total_combinations:,} possible combinations")
    
    n_iterations = min(100, total_combinations)
    print(f"Training {n_iterations} random combinations with 5-fold CV")
    
    start_time = time.time()
    
    try:
        # Use RandomizedSearchCV with 5-fold CV
        random_search = RandomizedSearchCV(
            ada_pipeline,
            param_dist,
            n_iter=n_iterations,
            cv=5,
            scoring='neg_mean_squared_error',
            n_jobs=-1,
            verbose=0,
            random_state=42,
            return_train_score=True,
            error_score='raise'
        )
        
        random_search.fit(X_train, y_train)
        
        elapsed = time.time() - start_time
        minutes = int(elapsed // 60)
        seconds = int(elapsed % 60)
        
        cv_rmse = np.sqrt(-random_search.best_score_)
        print(f"✓ Completed in {minutes}m {seconds}s")
        print(f"✓ Best CV RMSE: {cv_rmse:.4f}")
        print(f"✓ Best params: {random_search.best_params_}")
        
        if cv_rmse < best_score:
            best_score = cv_rmse
            best_scaler = scaler
            best_scaler_name = scaler_name
            best_params = random_search.best_params_
            best_model = random_search.best_estimator_
            print(f"✓ New overall best model found with {scaler_name}!")
            
    except Exception as e:
        print(f"✗ Error with {scaler_name}: {e}")
        continue

if best_model is None:
    print("\n❌ No model trained successfully. Using default AdaBoost.")
    best_model = Pipeline([
        ('scaler', StandardScaler()),
        ('ada', AdaBoostRegressor(
            n_estimators=100,
            learning_rate=1.0,
            loss='linear',
            random_state=42
        ))
    ])
    best_model.fit(X_train, y_train)
    best_scaler_name = "StandardScaler (fallback)"
    best_params = {"n_estimators": 100, "learning_rate": 1.0, "loss": "linear"}

print(f"\n{'='*60}")
print(f"BEST SCALER: {best_scaler_name}")
print(f"BEST PARAMETERS: {best_params}")
print(f"{'='*60}")

# Get predictions from best model
train_predictions = best_model.predict(X_train)
test_predictions = best_model.predict(X_test)

train_errors = train_predictions - y_train.values
test_errors = test_predictions - y_test.values

# Calculate ALL metrics
print("\n" + "="*80)
print("CALCULATING COMPREHENSIVE METRICS")
print("="*80)

metrics = {
    'R²': (r2_score, None),
    'RMSE': (lambda y, p: np.sqrt(mean_squared_error(y, p)), None),
    'MAE': (mean_absolute_error, None),
    'MedAE': (median_absolute_error, None),
    'Max Error': (max_error, None),
    'RAE': (relative_absolute_error, None),
    'RRSE': (root_relative_squared_error, None),
    'WI': (willmotts_index_of_agreement, None),
    'NSE': (nash_sutcliffe_efficiency, None),
    'MAPE (%)': (mean_absolute_percentage_error, None),
    'Explained Var': (explained_variance_score, None)
}

train_scores = {}
test_scores = {}
gaps = {}

for metric_name, (metric_func, _) in metrics.items():
    try:
        train_scores[metric_name] = metric_func(y_train, train_predictions)
        test_scores[metric_name] = metric_func(y_test, test_predictions)
        gaps[metric_name] = train_scores[metric_name] - test_scores[metric_name]
    except:
        train_scores[metric_name] = np.nan
        test_scores[metric_name] = np.nan
        gaps[metric_name] = np.nan

# ====================================================================
# COMPREHENSIVE RESULTS DISPLAY
# ====================================================================
print("\n" + "="*80)
print("COMPREHENSIVE PERFORMANCE METRICS")
print("="*80)

print("\n{:<20} {:<15} {:<15} {:<15}".format('Metric', 'Train', 'Test', 'Gap'))
print("-" * 65)

for metric_name in metrics.keys():
    if not np.isnan(train_scores[metric_name]):
        print("{:<20} {:<15.4f} {:<15.4f} {:<15.4f}".format(
            metric_name, train_scores[metric_name], test_scores[metric_name], gaps[metric_name]
        ))

# ====================================================================
# DETAILED PERFORMANCE GAP ANALYSIS
# ====================================================================
print("\n" + "="*80)
print("DETAILED PERFORMANCE GAP ANALYSIS")
print("="*80)

r2_gap = gaps.get('R²', 0)
rmse_gap = test_scores.get('RMSE', 0) - train_scores.get('RMSE', 0)
mae_gap = test_scores.get('MAE', 0) - train_scores.get('MAE', 0)

print(f"\nR² Gap (Train - Test):     {r2_gap:.4f}")
print(f"RMSE Gap (Test - Train):   {rmse_gap:.4f}")
print(f"MAE Gap (Test - Train):    {mae_gap:.4f}")

print("\n" + "-" * 60)
print("GAP INTERPRETATION:")
print("-" * 60)

if r2_gap > 0.05:
    print("✅ EXCELLENT: Strong regularization (Train > Test)")
    print("   → Model generalizes very well")
elif r2_gap > 0.02:
    print("✅ GOOD: Proper regularization (Train > Test)")
    print("   → Model generalizes well")
elif r2_gap > 0:
    print("⚠ ADEQUATE: Light regularization (Train slightly > Test)")
    print("   → Consider stronger regularization if needed")
elif abs(r2_gap) < 0.02:
    print("⚠ WARNING: Very similar performance")
    print("   → Model may be underfitting")
else:
    print("❌ PROBLEM: Test better than Train")
    print("   → Check for data leakage or split issues")

# ====================================================================
# CROSS-VALIDATION TABLE
# ====================================================================
print("\n" + "="*80)
print("CROSS-VALIDATION TABLE")
print("="*80)

try:
    # Perform 5-fold cross-validation on the best model
    kfold = KFold(n_splits=5, shuffle=True, random_state=42)
    cv_r2_scores = cross_val_score(best_model, X_train, y_train, cv=kfold, scoring='r2', n_jobs=-1)
    cv_rmse_scores = np.sqrt(-cross_val_score(best_model, X_train, y_train, cv=kfold, scoring='neg_mean_squared_error', n_jobs=-1))
    cv_mae_scores = -cross_val_score(best_model, X_train, y_train, cv=kfold, scoring='neg_mean_absolute_error', n_jobs=-1)
    
    # Create cross-validation table
    cv_table = pd.DataFrame({
        'Fold': range(1, 6),
        'R² Score': cv_r2_scores,
        'RMSE (%)': cv_rmse_scores,
        'MAE (%)': cv_mae_scores
    })
    
    # Add summary row
    summary_row = pd.DataFrame({
        'Fold': ['Mean ± Std'],
        'R² Score': [f"{cv_r2_scores.mean():.4f} ± {cv_r2_scores.std():.4f}"],
        'RMSE (%)': [f"{cv_rmse_scores.mean():.4f} ± {cv_rmse_scores.std():.4f}"],
        'MAE (%)': [f"{cv_mae_scores.mean():.4f} ± {cv_mae_scores.std():.4f}"]
    })
    
    cv_table_display = pd.concat([cv_table, summary_row], ignore_index=True)
    
    print("\n5-FOLD CROSS-VALIDATION RESULTS:")
    print(cv_table_display.to_string(index=False))
    
    # Save cross-validation table
    cv_table_display.to_csv(f"{results_dir}/Cross_Validation_Table.csv", index=False)
    print("\n✓ Cross_Validation_Table.csv saved")
    
    # Save detailed CV results
    cv_detailed = pd.DataFrame({
        'Fold': range(1, 6),
        'R²': cv_r2_scores,
        'RMSE': cv_rmse_scores,
        'MAE': cv_mae_scores
    })
    cv_detailed.to_csv(f"{results_dir}/Detailed_CV_Results.csv", index=False)
    print("✓ Detailed_CV_Results.csv saved")
    
except Exception as e:
    print(f"Could not compute cross-validation table: {e}")

# ====================================================================
# FEATURE IMPORTANCE ANALYSIS
# ====================================================================
print("\n" + "="*80)
print("FEATURE IMPORTANCE ANALYSIS")
print("="*80)

try:
    ada_model = best_model.named_steps['ada']
    
    # Get feature importance from AdaBoost ensemble
    if hasattr(ada_model.estimators_, '__len__'):
        feature_importances = np.zeros(X.shape[1])
        for estimator in ada_model.estimators_:
            if hasattr(estimator, 'feature_importances_'):
                feature_importances += estimator.feature_importances_
        feature_importances /= len(ada_model.estimators_)
        
        feature_importance_df = pd.DataFrame({
            'Feature': X.columns,
            'Importance': feature_importances
        }).sort_values('Importance', ascending=False)
        
        print("\nTOP 15 MOST IMPORTANT FEATURES:")
        print("-" * 70)
        for i, row in feature_importance_df.head(15).iterrows():
            print(f"{row['Feature']:50} {row['Importance']:.6f}")
        
        feature_importance_df.to_csv(f"{results_dir}/Feature_Importance.csv", index=False)
        print("\n✓ Feature_Importance.csv saved")
    else:
        print("\nCould not extract feature importance from the ensemble.")
        
except Exception as e:
    print(f"Could not extract feature importance: {e}")

# ====================================================================
# ADABOOST SPECIFIC ANALYSIS
# ====================================================================
print("\n" + "="*80)
print("ADABOOST SPECIFIC ANALYSIS")
print("="*80)

try:
    ada_model = best_model.named_steps['ada']
    
    print(f"\nAdaBoost Model Characteristics:")
    print(f"  - Base estimator: DecisionTreeRegressor")
    print(f"  - Number of estimators: {ada_model.n_estimators}")
    print(f"  - Learning rate: {ada_model.learning_rate}")
    print(f"  - Loss function: {ada_model.loss}")
    
    # Get estimator weights
    if hasattr(ada_model, 'estimator_weights_'):
        estimator_weights = ada_model.estimator_weights_
        print(f"\nEstimator Weights Statistics:")
        print(f"  - Mean weight: {np.mean(estimator_weights):.6f}")
        print(f"  - Std weight: {np.std(estimator_weights):.6f}")
        print(f"  - Max weight: {np.max(estimator_weights):.6f}")
        print(f"  - Min weight: {np.min(estimator_weights):.6f}")
        
        zero_weights = np.sum(estimator_weights == 0)
        print(f"  - Estimators with zero weight: {zero_weights}/{len(estimator_weights)}")
        
        # Save estimator weights
        weights_df = pd.DataFrame({
            'Estimator': range(1, len(estimator_weights) + 1),
            'Weight': estimator_weights
        })
        weights_df.to_csv(f"{results_dir}/Estimator_Weights.csv", index=False)
        print("\n✓ Estimator_Weights.csv saved")
    
    # Get estimator errors
    if hasattr(ada_model, 'estimator_errors_'):
        estimator_errors = ada_model.estimator_errors_
        print(f"\nEstimator Errors Statistics:")
        print(f"  - Mean error: {np.mean(estimator_errors):.6f}")
        print(f"  - Std error: {np.std(estimator_errors):.6f}")
        print(f"  - Max error: {np.max(estimator_errors):.6f}")
        print(f"  - Min error: {np.min(estimator_errors):.6f}")
        
        # Save estimator errors
        errors_df = pd.DataFrame({
            'Estimator': range(1, len(estimator_errors) + 1),
            'Error': estimator_errors
        })
        errors_df.to_csv(f"{results_dir}/Estimator_Errors.csv", index=False)
        print("\n✓ Estimator_Errors.csv saved")
        
except Exception as e:
    print(f"Could not extract AdaBoost details: {e}")

# ====================================================================
# SAVE ALL RESULTS
# ====================================================================
print("\n" + "="*80)
print("SAVING ALL RESULTS")
print("="*80)

# 1. Save predictions
train_results = pd.DataFrame({
    'TrueValues': y_train.values,
    'PredictedValues': train_predictions,
    'Error': train_errors,
    'AbsoluteError': np.abs(train_errors),
    'PercentageError': np.abs(train_errors / y_train.values) * 100
})
test_results = pd.DataFrame({
    'TrueValues': y_test.values,
    'PredictedValues': test_predictions,
    'Error': test_errors,
    'AbsoluteError': np.abs(test_errors),
    'PercentageError': np.abs(test_errors / y_test.values) * 100
})

train_results.to_csv(f"{results_dir}/Train_Results.csv", index=False)
test_results.to_csv(f"{results_dir}/Test_Results.csv", index=False)
print("✓ Train_Results.csv saved")
print("✓ Test_Results.csv saved")

# 2. Save all metrics
metrics_df = pd.DataFrame({
    'Metric': list(metrics.keys()),
    'Train': [train_scores[m] for m in metrics.keys()],
    'Test': [test_scores[m] for m in metrics.keys()],
    'Gap': [gaps[m] for m in metrics.keys()]
})
metrics_df.to_csv(f"{results_dir}/All_Metrics.csv", index=False)
print("✓ All_Metrics.csv saved")

# 3. Save best parameters
params_df = pd.DataFrame([best_params])
params_df.to_csv(f"{results_dir}/Best_Parameters.csv", index=False)
print("✓ Best_Parameters.csv saved")

# 4. Save model summary
with open(f"{results_dir}/Model_Summary.txt", 'w') as f:
    f.write("="*80 + "\n")
    f.write("ADABOOST MODEL SUMMARY - CALCIUM CARBONATE YIELD\n")
    f.write("="*80 + "\n\n")
    f.write(f"Target Variable: Calcium Carbonate Yield (%)\n")
    f.write(f"Dataset Size: {len(df)} samples\n")
    f.write(f"Features: {X.shape[1]}\n")
    f.write(f"Best Scaler: {best_scaler_name}\n\n")
    f.write("Best Parameters:\n")
    for key, value in best_params.items():
        f.write(f"  {key.replace('ada__', '').replace('estimator__', '')}: {value}\n")
    f.write("\n" + "="*40 + "\n")
    f.write("PERFORMANCE SUMMARY\n")
    f.write("="*40 + "\n\n")
    for metric_name in metrics.keys():
        if not np.isnan(train_scores[metric_name]):
            f.write(f"{metric_name:15} Train: {train_scores[metric_name]:10.4f}  Test: {test_scores[metric_name]:10.4f}  Gap: {gaps[metric_name]:10.4f}\n")
    f.write("\n" + "="*40 + "\n")
    f.write("CROSS-VALIDATION SUMMARY (5-Fold)\n")
    f.write("="*40 + "\n\n")
    f.write(f"Mean R²:  {cv_r2_scores.mean():.4f} ± {cv_r2_scores.std():.4f}\n")
    f.write(f"Mean RMSE: {cv_rmse_scores.mean():.4f} ± {cv_rmse_scores.std():.4f} %\n")
    f.write(f"Mean MAE:  {cv_mae_scores.mean():.4f} ± {cv_mae_scores.std():.4f} %\n")
print("✓ Model_Summary.txt saved")

# ====================================================================
# HIGH-RESOLUTION VISUALIZATIONS (500 DPI)
# ====================================================================
print("\n" + "="*80)
print("CREATING HIGH-RESOLUTION VISUALIZATIONS (500 DPI)")
print("="*80)

try:
    import matplotlib.pyplot as plt
    import seaborn as sns
    from scipy import stats as scipy_stats
    
    # Set high DPI for all figures
    plt.rcParams['figure.dpi'] = 500
    plt.rcParams['savefig.dpi'] = 500
    plt.rcParams['font.size'] = 10
    plt.rcParams['axes.labelsize'] = 12
    plt.rcParams['axes.titlesize'] = 14
    plt.rcParams['legend.fontsize'] = 10
    
    # Set style
    plt.style.use('seaborn-v0_8-darkgrid')
    sns.set_palette("husl")
    
    # Figure 1: Predictions vs Actual
    fig1, ax1 = plt.subplots(figsize=(10, 8))
    ax1.scatter(y_train, train_predictions, alpha=0.5, c='blue', s=30, 
                label=f'Train (R²={train_scores.get("R²", 0):.3f})')
    ax1.scatter(y_test, test_predictions, alpha=0.5, c='green', s=30, 
                label=f'Test (R²={test_scores.get("R²", 0):.3f})')
    
    # Add perfect prediction line
    min_val = min(y.min(), train_predictions.min(), test_predictions.min())
    max_val = max(y.max(), train_predictions.max(), test_predictions.max())
    ax1.plot([min_val, max_val], [min_val, max_val], 'r--', lw=2, label='Perfect Prediction')
    
    # Add ±10% error lines
    range_val = max_val - min_val
    ax1.plot([min_val, max_val], [min_val*0.9, max_val*0.9], 'k--', alpha=0.3, label='±10% Error')
    ax1.plot([min_val, max_val], [min_val*1.1, max_val*1.1], 'k--', alpha=0.3)
    
    ax1.set_xlabel('Actual Calcium Carbonate Yield (%)', fontsize=12, fontweight='bold')
    ax1.set_ylabel('Predicted Calcium Carbonate Yield (%)', fontsize=12, fontweight='bold')
    ax1.set_title('AdaBoost: Predicted vs Actual Calcium Carbonate Yield', fontsize=14, fontweight='bold')
    ax1.legend(loc='best', fontsize=10)
    ax1.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(f"{results_dir}/1_Predictions_vs_Actual.png", dpi=500, bbox_inches='tight')
    plt.close()
    print("✓ 1_Predictions_vs_Actual.png (500 DPI)")
    
    # Figure 2: Residual Analysis
    fig2, axes2 = plt.subplots(2, 2, figsize=(12, 10))
    
    # Residuals vs Predicted
    axes2[0,0].scatter(train_predictions, train_errors, alpha=0.5, c='blue', s=30, label='Train')
    axes2[0,0].scatter(test_predictions, test_errors, alpha=0.5, c='green', s=30, label='Test')
    axes2[0,0].axhline(y=0, color='r', linestyle='--', lw=2)
    axes2[0,0].set_xlabel('Predicted Values (%)')
    axes2[0,0].set_ylabel('Residuals (%)')
    axes2[0,0].set_title('Residual Plot')
    axes2[0,0].legend()
    axes2[0,0].grid(True, alpha=0.3)
    
    # Histogram of residuals
    axes2[0,1].hist(train_errors, bins=30, alpha=0.5, label='Train', density=True, edgecolor='black')
    axes2[0,1].hist(test_errors, bins=30, alpha=0.5, label='Test', density=True, edgecolor='black')
    axes2[0,1].set_xlabel('Residuals (%)')
    axes2[0,1].set_ylabel('Density')
    axes2[0,1].set_title('Residual Distribution')
    axes2[0,1].legend()
    axes2[0,1].grid(True, alpha=0.3)
    
    # Q-Q plot for residuals
    scipy_stats.probplot(test_errors, dist="norm", plot=axes2[1,0])
    axes2[1,0].set_title('Q-Q Plot (Test Residuals)')
    axes2[1,0].grid(True, alpha=0.3)
    
    # Residuals vs Actual
    axes2[1,1].scatter(y_test, test_errors, alpha=0.5, c='purple', s=30)
    axes2[1,1].axhline(y=0, color='r', linestyle='--', lw=2)
    axes2[1,1].set_xlabel('Actual Values (%)')
    axes2[1,1].set_ylabel('Residuals (%)')
    axes2[1,1].set_title('Residuals vs Actual Values')
    axes2[1,1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(f"{results_dir}/2_Residual_Analysis.png", dpi=500, bbox_inches='tight')
    plt.close()
    print("✓ 2_Residual_Analysis.png (500 DPI)")
    
    # Figure 3: Feature Importance
    if 'feature_importance_df' in locals():
        fig3, ax3 = plt.subplots(figsize=(12, 8))
        top15 = feature_importance_df.head(15)
        y_pos = np.arange(len(top15))
        bars = ax3.barh(y_pos, top15['Importance'].values, color='steelblue', edgecolor='black')
        ax3.set_yticks(y_pos)
        ax3.set_yticklabels(top15['Feature'].values, fontsize=9)
        ax3.set_xlabel('Importance', fontsize=12, fontweight='bold')
        ax3.set_title('Top 15 Most Important Features', fontsize=14, fontweight='bold')
        ax3.invert_yaxis()
        ax3.grid(True, alpha=0.3, axis='x')
        
        # Add value labels
        for i, bar in enumerate(bars):
            width = bar.get_width()
            ax3.text(width, bar.get_y() + bar.get_height()/2, f'{width:.4f}', 
                    ha='left', va='center', fontsize=8)
        
        plt.tight_layout()
        plt.savefig(f"{results_dir}/3_Feature_Importance.png", dpi=500, bbox_inches='tight')
        plt.close()
        print("✓ 3_Feature_Importance.png (500 DPI)")
    
    # Figure 4: Cross-Validation Performance
    if 'cv_r2_scores' in locals():
        fig4, ax4 = plt.subplots(figsize=(10, 6))
        folds = range(1, 6)
        ax4.plot(folds, cv_r2_scores, 'bo-', linewidth=2, markersize=8, label='R²')
        ax4.axhline(y=cv_r2_scores.mean(), color='b', linestyle='--', alpha=0.7, 
                   label=f'Mean R² = {cv_r2_scores.mean():.3f}')
        ax4.fill_between(folds, cv_r2_scores.mean() - cv_r2_scores.std(), 
                         cv_r2_scores.mean() + cv_r2_scores.std(), alpha=0.2, color='blue')
        
        ax4_twin = ax4.twinx()
        ax4_twin.plot(folds, cv_rmse_scores, 'ro-', linewidth=2, markersize=8, label='RMSE')
        ax4_twin.axhline(y=cv_rmse_scores.mean(), color='r', linestyle='--', alpha=0.7, 
                        label=f'Mean RMSE = {cv_rmse_scores.mean():.2f} %')
        
        ax4.set_xlabel('Fold', fontsize=12, fontweight='bold')
        ax4.set_ylabel('R² Score', fontsize=12, color='blue', fontweight='bold')
        ax4_twin.set_ylabel('RMSE (%)', fontsize=12, color='red', fontweight='bold')
        ax4.set_title('5-Fold Cross-Validation Performance', fontsize=14, fontweight='bold')
        ax4.grid(True, alpha=0.3)
        
        # Combine legends
        lines1, labels1 = ax4.get_legend_handles_labels()
        lines2, labels2 = ax4_twin.get_legend_handles_labels()
        ax4.legend(lines1 + lines2, labels1 + labels2, loc='best')
        
        plt.tight_layout()
        plt.savefig(f"{results_dir}/4_Cross_Validation_Performance.png", dpi=500, bbox_inches='tight')
        plt.close()
        print("✓ 4_Cross_Validation_Performance.png (500 DPI)")
    
    # Figure 5: Metrics Comparison
    fig5, ax5 = plt.subplots(figsize=(10, 6))
    metrics_to_plot = ['R²', 'RMSE', 'MAE', 'MedAE']
    x_pos = np.arange(len(metrics_to_plot))
    train_vals = [train_scores.get(m, 0) for m in metrics_to_plot]
    test_vals = [test_scores.get(m, 0) for m in metrics_to_plot]
    
    width = 0.35
    bars1 = ax5.bar(x_pos - width/2, train_vals, width, label='Train', alpha=0.8, color='blue', edgecolor='black')
    bars2 = ax5.bar(x_pos + width/2, test_vals, width, label='Test', alpha=0.8, color='green', edgecolor='black')
    
    # Add value labels on bars
    for bar in bars1:
        height = bar.get_height()
        ax5.text(bar.get_x() + bar.get_width()/2., height, f'{height:.3f}', 
                ha='center', va='bottom', fontsize=9, fontweight='bold')
    for bar in bars2:
        height = bar.get_height()
        ax5.text(bar.get_x() + bar.get_width()/2., height, f'{height:.3f}', 
                ha='center', va='bottom', fontsize=9, fontweight='bold')
    
    ax5.set_xticks(x_pos)
    ax5.set_xticklabels(metrics_to_plot, fontsize=11, fontweight='bold')
    ax5.set_ylabel('Score', fontsize=12, fontweight='bold')
    ax5.set_title('Train vs Test Performance Metrics', fontsize=14, fontweight='bold')
    ax5.legend(loc='best', fontsize=10)
    ax5.grid(True, alpha=0.3, axis='y')
    
    plt.tight_layout()
    plt.savefig(f"{results_dir}/5_Metrics_Comparison.png", dpi=500, bbox_inches='tight')
    plt.close()
    print("✓ 5_Metrics_Comparison.png (500 DPI)")
    
    # Figure 6: Estimator Weights
    if 'estimator_weights' in locals():
        fig6, ax6 = plt.subplots(figsize=(12, 6))
        ax6.plot(range(1, len(estimator_weights) + 1), estimator_weights, 'b-', linewidth=1, alpha=0.7)
        ax6.scatter(range(1, len(estimator_weights) + 1), estimator_weights, c='blue', s=20, alpha=0.5)
        ax6.set_xlabel('Estimator Number', fontsize=12, fontweight='bold')
        ax6.set_ylabel('Weight', fontsize=12, fontweight='bold')
        ax6.set_title('AdaBoost Estimator Weights', fontsize=14, fontweight='bold')
        ax6.axhline(y=estimator_weights.mean(), color='r', linestyle='--', alpha=0.5, 
                   label=f'Mean = {estimator_weights.mean():.4f}')
        ax6.legend()
        ax6.grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.savefig(f"{results_dir}/6_Estimator_Weights.png", dpi=500, bbox_inches='tight')
        plt.close()
        print("✓ 6_Estimator_Weights.png (500 DPI)")
    
    print("\n✓ All high-resolution visualizations (500 DPI) created successfully!")
    
except Exception as e:
    print(f"\n⚠ Could not create plots: {e}")
    print("Continuing with saved data files...")

# ====================================================================
# FINAL SUMMARY
# ====================================================================
print("\n" + "="*80)
print("FINAL SUMMARY")
print("="*80)

print(f"""
╔════════════════════════════════════════════════════════════════════════════╗
║                    ADABOOST MODEL - CALCIUM CARBONATE YIELD               ║
╚════════════════════════════════════════════════════════════════════════════╝

DATASET INFORMATION:
-------------------
• Dataset:              Calcium Carbonate Yield Prediction
• Target Variable:      Calcium Carbonate Yield (%)
• Total samples:        {len(df)}
• Features:             {X.shape[1]}
• Training samples:     {len(X_train)} (80%)
• Testing samples:      {len(X_test)} (20%)
• Best Scaler:          {best_scaler_name}

ADABOOST CHARACTERISTICS:
------------------------
• Base Estimator:       DecisionTreeRegressor
• Number of estimators: {best_params.get('ada__n_estimators', best_params.get('n_estimators', 'N/A'))}
• Learning rate:        {best_params.get('ada__learning_rate', best_params.get('learning_rate', 'N/A'))}
• Loss function:        {best_params.get('ada__loss', best_params.get('loss', 'N/A'))}
• Tree max depth:       {best_params.get('ada__estimator__max_depth', 'N/A')}

PERFORMANCE SUMMARY:
-------------------
• R² (Test):            {test_scores.get('R²', 0):.4f}
• RMSE (Test):          {test_scores.get('RMSE', 0):.4f} %
• MAE (Test):           {test_scores.get('MAE', 0):.4f} %
• MAPE (Test):          {test_scores.get('MAPE (%)', 0):.2f}%
• R² Gap:               {r2_gap:.4f}

CROSS-VALIDATION (5-Fold):
--------------------------
• Mean R²:              {cv_r2_scores.mean():.4f} ± {cv_r2_scores.std():.4f}
• Mean RMSE:            {cv_rmse_scores.mean():.4f} ± {cv_rmse_scores.std():.4f} %
• Mean MAE:             {cv_mae_scores.mean():.4f} ± {cv_mae_scores.std():.4f} %

╔════════════════════════════════════════════════════════════════════════════╗
║                           FILES SAVED (500 DPI)                            ║
╚════════════════════════════════════════════════════════════════════════════╝

Location: {results_dir}

CSV Files:
  ✓ All_Metrics.csv                 - All performance metrics
  ✓ Best_Parameters.csv              - Optimal hyperparameters
  ✓ Cross_Validation_Table.csv       - 5-fold CV results table
  ✓ Detailed_CV_Results.csv          - Detailed CV fold results
  ✓ Estimator_Errors.csv             - Individual estimator errors
  ✓ Estimator_Weights.csv            - Individual estimator weights
  ✓ Feature_Importance.csv           - Feature importance rankings
  ✓ Train_Results.csv                - Training set predictions
  ✓ Test_Results.csv                 - Testing set predictions
  ✓ Model_Summary.txt                - Complete model summary

High-Resolution Images (500 DPI):
  ✓ 1_Predictions_vs_Actual.png      - Scatter plot with error bounds
  ✓ 2_Residual_Analysis.png          - Comprehensive residual analysis
  ✓ 3_Feature_Importance.png         - Top 15 feature importance
  ✓ 4_Cross_Validation_Performance.png - CV performance visualization
  ✓ 5_Metrics_Comparison.png         - Train vs Test metrics comparison
  ✓ 6_Estimator_Weights.png          - AdaBoost estimator weights

""")

print("="*80)
print("PROGRAM COMPLETED SUCCESSFULLY!")
print(f"All results saved to: {results_dir}")
print("="*80)

LOADING DATA
Columns: ['cement type', 'CaO cement composition in RCP (wt.%)', 'liquid to solid ratio (ml/g)', 'CO2 flowrate (l/min/g)', 'CO2 concentration (volume fraction %)', 'carbonation time (min)', 'carbonation temperature (°C)', 'fineness (mm)', 'concentration of added Mg ions in solution (mol/l)', 'concentration of added Na ions in solution (mol/l)', 'concentration of added OH ions in solution (mol/l)', 'calcium carbonate yield (%)']
Cleaned columns: ['cement type', 'CaO cement composition in RCP (wt.%)', 'liquid to solid ratio (ml/g)', 'CO2 flowrate (l/min/g)', 'CO2 concentration (volume fraction %)', 'carbonation time (min)', 'carbonation temperature (°C)', 'fineness (mm)', 'concentration of added Mg ions in solution (mol/l)', 'concentration of added Na ions in solution (mol/l)', 'concentration of added OH ions in solution (mol/l)', 'calcium carbonate yield (%)']

ADVANCED DATA PREPROCESSING

1. Handling non-numeric columns...

2. Handling missing values...
Missing values befo

In [9]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import os
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import warnings
warnings.filterwarnings('ignore')

# Set the path to your results directory
results_dir = r"D:\2026 Work\My Papers\Predicting calcium carbonate yield\Models\AdaBoost"

# Load the saved results
print("="*80)
print("LOADING SAVED RESULTS")
print("="*80)

# Load predictions
train_results = pd.read_csv(f"{results_dir}/Train_Results.csv")
test_results = pd.read_csv(f"{results_dir}/Test_Results.csv")

# Load metrics
metrics_df = pd.read_csv(f"{results_dir}/All_Metrics.csv")

# Load feature importance if available
try:
    feature_importance = pd.read_csv(f"{results_dir}/Feature_Importance.csv")
    has_feature_importance = True
except:
    has_feature_importance = False

# Load cross-validation results
try:
    cv_results = pd.read_csv(f"{results_dir}/Detailed_CV_Results.csv")
    has_cv = True
except:
    has_cv = False

# Load estimator weights
try:
    estimator_weights = pd.read_csv(f"{results_dir}/Estimator_Weights.csv")
    has_weights = True
except:
    has_weights = False

print("✓ Data loaded successfully!")

# Create a new directory for enhanced plots
enhanced_dir = f"{results_dir}/Enhanced_Plots_White_Theme"
os.makedirs(enhanced_dir, exist_ok=True)

# Set WHITE THEME with THICK BORDERS
plt.rcParams['figure.dpi'] = 500
plt.rcParams['savefig.dpi'] = 500
plt.rcParams['font.size'] = 11
plt.rcParams['axes.labelsize'] = 12
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['legend.fontsize'] = 10
plt.rcParams['figure.figsize'] = (12, 8)

# White theme settings
plt.rcParams['axes.facecolor'] = 'white'
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['savefig.facecolor'] = 'white'
plt.rcParams['axes.edgecolor'] = 'black'
plt.rcParams['axes.linewidth'] = 2.5  # Thick border for axes
plt.rcParams['xtick.color'] = 'black'
plt.rcParams['ytick.color'] = 'black'
plt.rcParams['text.color'] = 'black'
plt.rcParams['axes.labelcolor'] = 'black'
plt.rcParams['grid.color'] = 'gray'
plt.rcParams['grid.linewidth'] = 0.8
plt.rcParams['lines.linewidth'] = 2

# Set style with white background
plt.style.use('default')
sns.set_style("whitegrid", {
    'grid.linestyle': '--',
    'grid.alpha': 0.3,
    'axes.edgecolor': 'black',
    'axes.linewidth': 2.5,
    'axes.facecolor': 'white',
    'figure.facecolor': 'white',
    'grid.color': 'gray'
})

# Custom color palette for white theme
colors = {
    'train': '#2E86AB',  # Blue
    'test': '#A23B72',   # Purple
    'line': '#000000',   # Black
    'error': '#DC143C',  # Crimson
    'grid': '#D3D3D3'    # Light gray
}

print("\n" + "="*80)
print("CREATING ENHANCED VISUALIZATIONS (WHITE THEME + THICK BORDERS)")
print("="*80)

# ============================================================================
# PLOT 1: Enhanced Predictions vs Actual with Confidence Intervals
# ============================================================================
print("\n1. Creating Enhanced Predictions vs Actual Plot...")

fig1, ax1 = plt.subplots(figsize=(12, 8))

# Extract data
y_train_true = train_results['TrueValues'].values
y_train_pred = train_results['PredictedValues'].values
y_test_true = test_results['TrueValues'].values
y_test_pred = test_results['PredictedValues'].values

# Calculate R² values
r2_train = r2_score(y_train_true, y_train_pred)
r2_test = r2_score(y_test_true, y_test_pred)

# Create scatter plots with different colors and alphas
ax1.scatter(y_train_true, y_train_pred, alpha=0.6, c=colors['train'], s=50, 
            edgecolors='black', linewidth=1.5, label=f'Training Set (R² = {r2_train:.4f})')
ax1.scatter(y_test_true, y_test_pred, alpha=0.6, c=colors['test'], s=50, 
            edgecolors='black', linewidth=1.5, label=f'Testing Set (R² = {r2_test:.4f})')

# Perfect prediction line
min_val = min(y_train_true.min(), y_test_true.min(), y_train_pred.min(), y_test_pred.min())
max_val = max(y_train_true.max(), y_test_true.max(), y_train_pred.max(), y_test_pred.max())
ax1.plot([min_val, max_val], [min_val, max_val], 'k-', lw=3, label='Perfect Prediction', alpha=0.8)

# Add ±10% error bounds
ax1.plot([min_val, max_val], [min_val*0.9, max_val*0.9], 'r--', lw=2, alpha=0.7, label='±10% Error')
ax1.plot([min_val, max_val], [min_val*1.1, max_val*1.1], 'r--', lw=2, alpha=0.7)

# Add ±20% error bounds
ax1.plot([min_val, max_val], [min_val*0.8, max_val*0.8], 'g--', lw=1.5, alpha=0.5, label='±20% Error')
ax1.plot([min_val, max_val], [min_val*1.2, max_val*1.2], 'g--', lw=1.5, alpha=0.5)

# Fill between error bounds
ax1.fill_between([min_val, max_val], [min_val*0.9, max_val*0.9], [min_val*1.1, max_val*1.1], 
                  alpha=0.1, color='red', label='±10% Band')
ax1.fill_between([min_val, max_val], [min_val*0.8, max_val*0.8], [min_val*1.2, max_val*1.2], 
                  alpha=0.05, color='green', label='±20% Band')

ax1.set_xlabel('Actual Calcium Carbonate Yield (%)', fontsize=13, fontweight='bold')
ax1.set_ylabel('Predicted Calcium Carbonate Yield (%)', fontsize=13, fontweight='bold')
ax1.set_title('AdaBoost: Predicted vs Actual Calcium Carbonate Yield\nwith Error Bounds', 
              fontsize=15, fontweight='bold', pad=20)
ax1.legend(loc='best', fontsize=10, framealpha=0.9, edgecolor='black', fancybox=False, shadow=False)
ax1.grid(True, alpha=0.3, linestyle='--', linewidth=0.8)
ax1.set_aspect('equal', adjustable='box')
ax1.tick_params(axis='both', which='major', labelsize=10, width=1.5, length=6)
for spine in ax1.spines.values():
    spine.set_linewidth(2.5)
    spine.set_color('black')

# Add text box with statistics
stats_text = f'Training: R² = {r2_train:.4f}\nTesting: R² = {r2_test:.4f}\n'
stats_text += f'RMSE (Test) = {metrics_df[metrics_df["Metric"]=="RMSE"]["Test"].values[0]:.4f} %\n'
stats_text += f'MAE (Test) = {metrics_df[metrics_df["Metric"]=="MAE"]["Test"].values[0]:.4f} %'
ax1.text(0.05, 0.95, stats_text, transform=ax1.transAxes, fontsize=10,
         verticalalignment='top', bbox=dict(boxstyle='round', facecolor='white', 
                                           edgecolor='black', linewidth=2))

plt.tight_layout()
plt.savefig(f"{enhanced_dir}/1_Enhanced_Predictions_vs_Actual.png", dpi=500, bbox_inches='tight', facecolor='white')
plt.close()
print("   ✓ Saved: 1_Enhanced_Predictions_vs_Actual.png")

# ============================================================================
# PLOT 2: Comprehensive Residual Analysis with Statistical Tests
# ============================================================================
print("\n2. Creating Comprehensive Residual Analysis...")

fig2, axes2 = plt.subplots(2, 3, figsize=(15, 10))

# Calculate residuals
train_residuals = train_results['Error'].values
test_residuals = test_results['Error'].values

# 2.1 Residuals vs Predicted
axes2[0,0].scatter(train_results['PredictedValues'], train_residuals, alpha=0.5, 
                    c=colors['train'], s=40, edgecolors='black', linewidth=1, label='Training')
axes2[0,0].scatter(test_results['PredictedValues'], test_residuals, alpha=0.5, 
                    c=colors['test'], s=40, edgecolors='black', linewidth=1, label='Testing')
axes2[0,0].axhline(y=0, color='red', linestyle='--', lw=2.5)
axes2[0,0].set_xlabel('Predicted Values (%)', fontsize=11, fontweight='bold')
axes2[0,0].set_ylabel('Residuals (%)', fontsize=11, fontweight='bold')
axes2[0,0].set_title('Residuals vs Predicted Values', fontsize=12, fontweight='bold')
axes2[0,0].legend(frameon=True, edgecolor='black', fancybox=False)
axes2[0,0].grid(True, alpha=0.3, linestyle='--')
axes2[0,0].tick_params(width=1.5, length=5)
for spine in axes2[0,0].spines.values():
    spine.set_linewidth(2.5)

# 2.2 Histogram of Residuals with KDE
axes2[0,1].hist(train_residuals, bins=30, alpha=0.5, density=True, 
                 label='Training', color=colors['train'], edgecolor='black', linewidth=1.5)
axes2[0,1].hist(test_residuals, bins=30, alpha=0.5, density=True, 
                 label='Testing', color=colors['test'], edgecolor='black', linewidth=1.5)
# Add KDE
from scipy.stats import gaussian_kde
if len(test_residuals) > 1:
    kde = gaussian_kde(test_residuals)
    x_range = np.linspace(test_residuals.min(), test_residuals.max(), 100)
    axes2[0,1].plot(x_range, kde(x_range), 'r-', lw=2.5, label='KDE (Test)')
axes2[0,1].set_xlabel('Residuals (%)', fontsize=11, fontweight='bold')
axes2[0,1].set_ylabel('Density', fontsize=11, fontweight='bold')
axes2[0,1].set_title('Residual Distribution', fontsize=12, fontweight='bold')
axes2[0,1].legend(frameon=True, edgecolor='black', fancybox=False)
axes2[0,1].grid(True, alpha=0.3, linestyle='--')
axes2[0,1].tick_params(width=1.5, length=5)
for spine in axes2[0,1].spines.values():
    spine.set_linewidth(2.5)

# 2.3 Q-Q Plot for Test Residuals
stats.probplot(test_residuals, dist="norm", plot=axes2[0,2])
axes2[0,2].set_title('Q-Q Plot (Test Residuals)', fontsize=12, fontweight='bold')
axes2[0,2].grid(True, alpha=0.3, linestyle='--')
axes2[0,2].tick_params(width=1.5, length=5)
for spine in axes2[0,2].spines.values():
    spine.set_linewidth(2.5)
# Change line colors to black
axes2[0,2].get_lines()[0].set_color('blue')
axes2[0,2].get_lines()[0].set_linewidth(2)

# 2.4 Residuals vs Actual Values
axes2[1,0].scatter(test_results['TrueValues'], test_residuals, alpha=0.5, 
                    c='purple', s=40, edgecolors='black', linewidth=1.5)
axes2[1,0].axhline(y=0, color='red', linestyle='--', lw=2.5)
axes2[1,0].set_xlabel('Actual Values (%)', fontsize=11, fontweight='bold')
axes2[1,0].set_ylabel('Residuals (%)', fontsize=11, fontweight='bold')
axes2[1,0].set_title('Residuals vs Actual Values', fontsize=12, fontweight='bold')
axes2[1,0].grid(True, alpha=0.3, linestyle='--')
axes2[1,0].tick_params(width=1.5, length=5)
for spine in axes2[1,0].spines.values():
    spine.set_linewidth(2.5)

# 2.5 Box Plot of Residuals
residuals_data = [train_residuals, test_residuals]
bp = axes2[1,1].boxplot(residuals_data, labels=['Training', 'Testing'], 
                         patch_artist=True, showmeans=True, widths=0.6)
for box in bp['boxes']:
    box.set_facecolor(colors['train'])
    box.set_edgecolor('black')
    box.set_linewidth(2.5)
for median in bp['medians']:
    median.set_color('black')
    median.set_linewidth(2.5)
for mean in bp['means']:
    mean.set_marker('D')
    mean.set_markerfacecolor('red')
    mean.set_markeredgecolor('black')
    mean.set_markersize(8)
for whisker in bp['whiskers']:
    whisker.set_linewidth(2)
for cap in bp['caps']:
    cap.set_linewidth(2)
axes2[1,1].set_ylabel('Residuals (%)', fontsize=11, fontweight='bold')
axes2[1,1].set_title('Residual Box Plot', fontsize=12, fontweight='bold')
axes2[1,1].grid(True, alpha=0.3, axis='y', linestyle='--')
axes2[1,1].tick_params(width=1.5, length=5)
for spine in axes2[1,1].spines.values():
    spine.set_linewidth(2.5)

# 2.6 Statistical Summary Text
stats_text = f'Residual Statistics:\n\n'
stats_text += f'Training:\n'
stats_text += f'  Mean: {np.mean(train_residuals):.4f}\n'
stats_text += f'  Std: {np.std(train_residuals):.4f}\n'
stats_text += f'  Skew: {stats.skew(train_residuals):.4f}\n\n'
stats_text += f'Testing:\n'
stats_text += f'  Mean: {np.mean(test_residuals):.4f}\n'
stats_text += f'  Std: {np.std(test_residuals):.4f}\n'
stats_text += f'  Skew: {stats.skew(test_residuals):.4f}'

axes2[1,2].text(0.1, 0.5, stats_text, transform=axes2[1,2].transAxes, fontsize=10,
                verticalalignment='center', bbox=dict(boxstyle='round', facecolor='white', 
                                                     edgecolor='black', linewidth=2.5))
axes2[1,2].axis('off')

plt.suptitle('Comprehensive Residual Analysis', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(f"{enhanced_dir}/2_Comprehensive_Residual_Analysis.png", dpi=500, bbox_inches='tight', facecolor='white')
plt.close()
print("   ✓ Saved: 2_Comprehensive_Residual_Analysis.png")

# ============================================================================
# PLOT 3: Feature Importance Heatmap and Bar Chart
# ============================================================================
if has_feature_importance:
    print("\n3. Creating Feature Importance Visualizations...")
    
    fig3, axes3 = plt.subplots(1, 2, figsize=(15, 8))
    
    # 3.1 Top 20 Features Bar Chart
    top20 = feature_importance.head(20)
    colors_bar = plt.cm.viridis(np.linspace(0, 1, len(top20)))
    bars = axes3[0].barh(range(len(top20)), top20['Importance'].values, 
                         color=colors_bar, edgecolor='black', linewidth=2)
    axes3[0].set_yticks(range(len(top20)))
    axes3[0].set_yticklabels(top20['Feature'].values, fontsize=9)
    axes3[0].set_xlabel('Importance', fontsize=12, fontweight='bold')
    axes3[0].set_title('Top 20 Most Important Features', fontsize=13, fontweight='bold')
    axes3[0].invert_yaxis()
    axes3[0].grid(True, alpha=0.3, axis='x', linestyle='--')
    axes3[0].tick_params(width=1.5, length=5)
    for spine in axes3[0].spines.values():
        spine.set_linewidth(2.5)
    
    # Add value labels
    for i, bar in enumerate(bars):
        width = bar.get_width()
        axes3[0].text(width, bar.get_y() + bar.get_height()/2, f'{width:.4f}', 
                      ha='left', va='center', fontsize=8, fontweight='bold')
    
    # 3.2 Cumulative Importance
    cumulative_importance = feature_importance['Importance'].cumsum()
    axes3[1].plot(range(1, len(cumulative_importance) + 1), cumulative_importance, 
                  'b-o', linewidth=2.5, markersize=6, markerfacecolor='blue', 
                  markeredgecolor='black', markeredgewidth=1.5)
    axes3[1].axhline(y=0.8, color='red', linestyle='--', lw=2.5, alpha=0.7, label='80% Threshold')
    axes3[1].axhline(y=0.9, color='green', linestyle='--', lw=2.5, alpha=0.7, label='90% Threshold')
    axes3[1].set_xlabel('Number of Features', fontsize=12, fontweight='bold')
    axes3[1].set_ylabel('Cumulative Importance', fontsize=12, fontweight='bold')
    axes3[1].set_title('Cumulative Feature Importance', fontsize=13, fontweight='bold')
    axes3[1].legend(frameon=True, edgecolor='black', fancybox=False)
    axes3[1].grid(True, alpha=0.3, linestyle='--')
    axes3[1].tick_params(width=1.5, length=5)
    for spine in axes3[1].spines.values():
        spine.set_linewidth(2.5)
    
    # Find number of features needed for 80% and 90%
    n_80 = np.where(cumulative_importance >= 0.8)[0][0] + 1 if any(cumulative_importance >= 0.8) else len(cumulative_importance)
    n_90 = np.where(cumulative_importance >= 0.9)[0][0] + 1 if any(cumulative_importance >= 0.9) else len(cumulative_importance)
    axes3[1].axvline(x=n_80, ymin=0, ymax=0.8, color='red', linestyle=':', alpha=0.5, linewidth=2)
    axes3[1].axvline(x=n_90, ymin=0, ymax=0.9, color='green', linestyle=':', alpha=0.5, linewidth=2)
    
    # Add text annotations
    axes3[1].text(n_80, 0.75, f'{n_80} features\n(80%)', ha='center', fontsize=9, 
                  bbox=dict(boxstyle='round', facecolor='white', edgecolor='black'))
    axes3[1].text(n_90, 0.85, f'{n_90} features\n(90%)', ha='center', fontsize=9,
                  bbox=dict(boxstyle='round', facecolor='white', edgecolor='black'))
    
    plt.suptitle('Feature Importance Analysis', fontsize=16, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.savefig(f"{enhanced_dir}/3_Feature_Importance_Analysis.png", dpi=500, bbox_inches='tight', facecolor='white')
    plt.close()
    print("   ✓ Saved: 3_Feature_Importance_Analysis.png")

# ============================================================================
# PLOT 4: Cross-Validation Performance with Error Bars
# ============================================================================
if has_cv:
    print("\n4. Creating Cross-Validation Performance Plot...")
    
    fig4, axes4 = plt.subplots(1, 2, figsize=(14, 6))
    
    # 4.1 CV Scores with Error Bars
    folds = cv_results['Fold'].values
    r2_scores = cv_results['R²'].values
    rmse_scores = cv_results['RMSE'].values
    
    # R² plot
    bars1 = axes4[0].bar(folds, r2_scores, color='steelblue', alpha=0.7, 
                         edgecolor='black', linewidth=2)
    axes4[0].axhline(y=r2_scores.mean(), color='red', linestyle='--', lw=2.5, 
                     label=f'Mean R² = {r2_scores.mean():.4f}')
    axes4[0].fill_between(folds, r2_scores.mean() - r2_scores.std(), 
                           r2_scores.mean() + r2_scores.std(), alpha=0.2, color='red')
    axes4[0].set_xlabel('Fold', fontsize=12, fontweight='bold')
    axes4[0].set_ylabel('R² Score', fontsize=12, fontweight='bold')
    axes4[0].set_title('5-Fold Cross-Validation R² Scores', fontsize=13, fontweight='bold')
    axes4[0].legend(frameon=True, edgecolor='black', fancybox=False)
    axes4[0].grid(True, alpha=0.3, axis='y', linestyle='--')
    axes4[0].tick_params(width=1.5, length=5)
    for spine in axes4[0].spines.values():
        spine.set_linewidth(2.5)
    
    # Add value labels on bars
    for bar in bars1:
        height = bar.get_height()
        axes4[0].text(bar.get_x() + bar.get_width()/2., height + 0.01, f'{height:.4f}', 
                      ha='center', va='bottom', fontsize=9, fontweight='bold')
    
    # RMSE plot
    bars2 = axes4[1].bar(folds, rmse_scores, color='coral', alpha=0.7, 
                         edgecolor='black', linewidth=2)
    axes4[1].axhline(y=rmse_scores.mean(), color='red', linestyle='--', lw=2.5,
                     label=f'Mean RMSE = {rmse_scores.mean():.4f} %')
    axes4[1].fill_between(folds, rmse_scores.mean() - rmse_scores.std(), 
                           rmse_scores.mean() + rmse_scores.std(), alpha=0.2, color='red')
    axes4[1].set_xlabel('Fold', fontsize=12, fontweight='bold')
    axes4[1].set_ylabel('RMSE (%)', fontsize=12, fontweight='bold')
    axes4[1].set_title('5-Fold Cross-Validation RMSE', fontsize=13, fontweight='bold')
    axes4[1].legend(frameon=True, edgecolor='black', fancybox=False)
    axes4[1].grid(True, alpha=0.3, axis='y', linestyle='--')
    axes4[1].tick_params(width=1.5, length=5)
    for spine in axes4[1].spines.values():
        spine.set_linewidth(2.5)
    
    # Add value labels on bars
    for bar in bars2:
        height = bar.get_height()
        axes4[1].text(bar.get_x() + bar.get_width()/2., height + 0.2, f'{height:.2f}', 
                      ha='center', va='bottom', fontsize=9, fontweight='bold')
    
    plt.suptitle('Cross-Validation Performance Analysis', fontsize=16, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.savefig(f"{enhanced_dir}/4_Cross_Validation_Performance.png", dpi=500, bbox_inches='tight', facecolor='white')
    plt.close()
    print("   ✓ Saved: 4_Cross_Validation_Performance.png")

# ============================================================================
# PLOT 5: Metrics Radar Chart
# ============================================================================
print("\n5. Creating Metrics Radar Chart...")

# Prepare data for radar chart
metrics_radar = ['R²', 'NSE', 'WI', 'Explained Var']
train_values = [metrics_df[metrics_df['Metric']==m]['Train'].values[0] for m in metrics_radar]
test_values = [metrics_df[metrics_df['Metric']==m]['Test'].values[0] for m in metrics_radar]

# Normalize values for radar chart (0-1 scale)
train_norm = [max(0, min(1, v)) for v in train_values]
test_norm = [max(0, min(1, v)) for v in test_values]

# Create radar chart
angles = np.linspace(0, 2*np.pi, len(metrics_radar), endpoint=False).tolist()
train_norm += train_norm[:1]
test_norm += test_norm[:1]
angles += angles[:1]

fig5, ax5 = plt.subplots(figsize=(10, 10), subplot_kw=dict(projection='polar'))
ax5.plot(angles, train_norm, 'o-', linewidth=3, label='Training', color=colors['train'], 
         markersize=8, markerfacecolor=colors['train'], markeredgecolor='black', markeredgewidth=1.5)
ax5.fill(angles, train_norm, alpha=0.25, color=colors['train'])
ax5.plot(angles, test_norm, 'o-', linewidth=3, label='Testing', color=colors['test'],
         markersize=8, markerfacecolor=colors['test'], markeredgecolor='black', markeredgewidth=1.5)
ax5.fill(angles, test_norm, alpha=0.25, color=colors['test'])
ax5.set_xticks(angles[:-1])
ax5.set_xticklabels(metrics_radar, fontsize=11, fontweight='bold')
ax5.set_ylim(0, 1)
ax5.set_yticks([0.2, 0.4, 0.6, 0.8, 1.0])
ax5.set_yticklabels(['0.2', '0.4', '0.6', '0.8', '1.0'], fontsize=9)
ax5.set_title('Model Performance Radar Chart', fontsize=15, fontweight='bold', pad=20)
ax5.legend(loc='upper right', bbox_to_anchor=(1.1, 1.1), frameon=True, edgecolor='black', fancybox=False)
ax5.grid(True, linestyle='--', linewidth=1.5)
ax5.set_facecolor('white')
for spine in ax5.spines.values():
    spine.set_linewidth(2.5)

plt.tight_layout()
plt.savefig(f"{enhanced_dir}/5_Metrics_Radar_Chart.png", dpi=500, bbox_inches='tight', facecolor='white')
plt.close()
print("   ✓ Saved: 5_Metrics_Radar_Chart.png")

# ============================================================================
# PLOT 6: Error Distribution with Statistical Tests
# ============================================================================
print("\n6. Creating Error Distribution Analysis...")

fig6, axes6 = plt.subplots(2, 2, figsize=(14, 10))

# 6.1 Absolute Error Distribution
train_abs_error = train_results['AbsoluteError'].values
test_abs_error = test_results['AbsoluteError'].values

axes6[0,0].hist(train_abs_error, bins=30, alpha=0.5, density=True, 
                 label='Training', color=colors['train'], edgecolor='black', linewidth=1.5)
axes6[0,0].hist(test_abs_error, bins=30, alpha=0.5, density=True, 
                 label='Testing', color=colors['test'], edgecolor='black', linewidth=1.5)
axes6[0,0].set_xlabel('Absolute Error (%)', fontsize=11, fontweight='bold')
axes6[0,0].set_ylabel('Density', fontsize=11, fontweight='bold')
axes6[0,0].set_title('Absolute Error Distribution', fontsize=12, fontweight='bold')
axes6[0,0].legend(frameon=True, edgecolor='black', fancybox=False)
axes6[0,0].grid(True, alpha=0.3, linestyle='--')
axes6[0,0].tick_params(width=1.5, length=5)
for spine in axes6[0,0].spines.values():
    spine.set_linewidth(2.5)

# 6.2 Percentage Error Distribution
train_pct_error = train_results['PercentageError'].values
test_pct_error = test_results['PercentageError'].values

axes6[0,1].hist(train_pct_error, bins=30, alpha=0.5, density=True, 
                 label='Training', color=colors['train'], edgecolor='black', linewidth=1.5)
axes6[0,1].hist(test_pct_error, bins=30, alpha=0.5, density=True, 
                 label='Testing', color=colors['test'], edgecolor='black', linewidth=1.5)
axes6[0,1].set_xlabel('Percentage Error (%)', fontsize=11, fontweight='bold')
axes6[0,1].set_ylabel('Density', fontsize=11, fontweight='bold')
axes6[0,1].set_title('Percentage Error Distribution', fontsize=12, fontweight='bold')
axes6[0,1].legend(frameon=True, edgecolor='black', fancybox=False)
axes6[0,1].grid(True, alpha=0.3, linestyle='--')
axes6[0,1].tick_params(width=1.5, length=5)
for spine in axes6[0,1].spines.values():
    spine.set_linewidth(2.5)

# 6.3 Cumulative Error Distribution
sorted_test_abs = np.sort(test_abs_error)
cumulative = np.arange(1, len(sorted_test_abs) + 1) / len(sorted_test_abs)
axes6[1,0].plot(sorted_test_abs, cumulative, 'b-', linewidth=2.5, label='Testing')
axes6[1,0].axhline(y=0.5, color='red', linestyle='--', lw=2.5, alpha=0.7, label='Median')
axes6[1,0].axhline(y=0.9, color='green', linestyle='--', lw=2.5, alpha=0.7, label='90th Percentile')
axes6[1,0].set_xlabel('Absolute Error (%)', fontsize=11, fontweight='bold')
axes6[1,0].set_ylabel('Cumulative Probability', fontsize=11, fontweight='bold')
axes6[1,0].set_title('Cumulative Error Distribution', fontsize=12, fontweight='bold')
axes6[1,0].legend(frameon=True, edgecolor='black', fancybox=False)
axes6[1,0].grid(True, alpha=0.3, linestyle='--')
axes6[1,0].tick_params(width=1.5, length=5)
for spine in axes6[1,0].spines.values():
    spine.set_linewidth(2.5)

# 6.4 Error Statistics Table
stats_text = f'Error Statistics:\n\n'
stats_text += f'Training Set:\n'
stats_text += f'  MAE: {np.mean(train_abs_error):.4f} %\n'
stats_text += f'  MedAE: {np.median(train_abs_error):.4f} %\n'
stats_text += f'  Max Error: {np.max(train_abs_error):.4f} %\n'
stats_text += f'  MAPE: {np.mean(train_pct_error):.2f}%\n\n'
stats_text += f'Testing Set:\n'
stats_text += f'  MAE: {np.mean(test_abs_error):.4f} %\n'
stats_text += f'  MedAE: {np.median(test_abs_error):.4f} %\n'
stats_text += f'  Max Error: {np.max(test_abs_error):.4f} %\n'
stats_text += f'  MAPE: {np.mean(test_pct_error):.2f}%'

axes6[1,1].text(0.1, 0.5, stats_text, transform=axes6[1,1].transAxes, fontsize=10,
                verticalalignment='center', bbox=dict(boxstyle='round', facecolor='white', 
                                                     edgecolor='black', linewidth=2.5))
axes6[1,1].axis('off')

plt.suptitle('Error Distribution Analysis', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(f"{enhanced_dir}/6_Error_Distribution_Analysis.png", dpi=500, bbox_inches='tight', facecolor='white')
plt.close()
print("   ✓ Saved: 6_Error_Distribution_Analysis.png")

# ============================================================================
# PLOT 7: Estimator Weights Analysis (if available)
# ============================================================================
if has_weights:
    print("\n7. Creating Estimator Weights Analysis...")
    
    fig7, axes7 = plt.subplots(1, 2, figsize=(14, 6))
    
    weights = estimator_weights['Weight'].values
    
    # 7.1 Estimator Weights Over Time
    axes7[0].plot(range(1, len(weights) + 1), weights, 'b-', linewidth=2.5, alpha=0.7)
    axes7[0].scatter(range(1, len(weights) + 1), weights, c='blue', s=30, alpha=0.6,
                     edgecolors='black', linewidth=1.5)
    axes7[0].axhline(y=weights.mean(), color='red', linestyle='--', lw=2.5, alpha=0.7, 
                     label=f'Mean = {weights.mean():.4f}')
    axes7[0].set_xlabel('Estimator Number', fontsize=12, fontweight='bold')
    axes7[0].set_ylabel('Weight', fontsize=12, fontweight='bold')
    axes7[0].set_title('AdaBoost Estimator Weights', fontsize=13, fontweight='bold')
    axes7[0].legend(frameon=True, edgecolor='black', fancybox=False)
    axes7[0].grid(True, alpha=0.3, linestyle='--')
    axes7[0].tick_params(width=1.5, length=5)
    for spine in axes7[0].spines.values():
        spine.set_linewidth(2.5)
    
    # 7.2 Weight Distribution
    axes7[1].hist(weights, bins=30, color='steelblue', alpha=0.7, edgecolor='black', linewidth=1.5)
    axes7[1].axvline(x=weights.mean(), color='red', linestyle='--', lw=2.5, 
                     label=f'Mean = {weights.mean():.4f}')
    axes7[1].axvline(x=np.median(weights), color='green', linestyle='--', lw=2.5, 
                     label=f'Median = {np.median(weights):.4f}')
    axes7[1].set_xlabel('Weight', fontsize=12, fontweight='bold')
    axes7[1].set_ylabel('Frequency', fontsize=12, fontweight='bold')
    axes7[1].set_title('Distribution of Estimator Weights', fontsize=13, fontweight='bold')
    axes7[1].legend(frameon=True, edgecolor='black', fancybox=False)
    axes7[1].grid(True, alpha=0.3, axis='y', linestyle='--')
    axes7[1].tick_params(width=1.5, length=5)
    for spine in axes7[1].spines.values():
        spine.set_linewidth(2.5)
    
    plt.suptitle('AdaBoost Estimator Analysis', fontsize=16, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.savefig(f"{enhanced_dir}/7_Estimator_Weights_Analysis.png", dpi=500, bbox_inches='tight', facecolor='white')
    plt.close()
    print("   ✓ Saved: 7_Estimator_Weights_Analysis.png")

# ============================================================================
# PLOT 8: Performance Metrics Dashboard
# ============================================================================
print("\n8. Creating Performance Metrics Dashboard...")

fig8, axes8 = plt.subplots(2, 2, figsize=(14, 10))

# 8.1 Metrics Bar Chart
metrics_to_show = ['R²', 'NSE', 'WI', 'Explained Var']
train_metrics = [metrics_df[metrics_df['Metric']==m]['Train'].values[0] for m in metrics_to_show]
test_metrics = [metrics_df[metrics_df['Metric']==m]['Test'].values[0] for m in metrics_to_show]

x_pos = np.arange(len(metrics_to_show))
width = 0.35
bars1 = axes8[0,0].bar(x_pos - width/2, train_metrics, width, label='Training', 
                        color=colors['train'], alpha=0.8, edgecolor='black', linewidth=2)
bars2 = axes8[0,0].bar(x_pos + width/2, test_metrics, width, label='Testing', 
                        color=colors['test'], alpha=0.8, edgecolor='black', linewidth=2)
axes8[0,0].set_xticks(x_pos)
axes8[0,0].set_xticklabels(metrics_to_show, fontsize=11)
axes8[0,0].set_ylabel('Score', fontsize=11, fontweight='bold')
axes8[0,0].set_title('Performance Metrics Comparison', fontsize=12, fontweight='bold')
axes8[0,0].legend(frameon=True, edgecolor='black', fancybox=False)
axes8[0,0].grid(True, alpha=0.3, axis='y', linestyle='--')
axes8[0,0].tick_params(width=1.5, length=5)
for spine in axes8[0,0].spines.values():
    spine.set_linewidth(2.5)

# Add value labels
for bar in bars1:
    height = bar.get_height()
    axes8[0,0].text(bar.get_x() + bar.get_width()/2., height, f'{height:.3f}', 
                    ha='center', va='bottom', fontsize=9, fontweight='bold')
for bar in bars2:
    height = bar.get_height()
    axes8[0,0].text(bar.get_x() + bar.get_width()/2., height, f'{height:.3f}', 
                    ha='center', va='bottom', fontsize=9, fontweight='bold')

# 8.2 Error Metrics Comparison
error_metrics = ['RMSE', 'MAE', 'MedAE', 'Max Error']
train_errors = [metrics_df[metrics_df['Metric']==m]['Train'].values[0] for m in error_metrics]
test_errors = [metrics_df[metrics_df['Metric']==m]['Test'].values[0] for m in error_metrics]

x_pos2 = np.arange(len(error_metrics))
bars3 = axes8[0,1].bar(x_pos2 - width/2, train_errors, width, label='Training', 
                        color=colors['train'], alpha=0.8, edgecolor='black', linewidth=2)
bars4 = axes8[0,1].bar(x_pos2 + width/2, test_errors, width, label='Testing', 
                        color=colors['test'], alpha=0.8, edgecolor='black', linewidth=2)
axes8[0,1].set_xticks(x_pos2)
axes8[0,1].set_xticklabels(error_metrics, fontsize=11)
axes8[0,1].set_ylabel('Error (%)', fontsize=11, fontweight='bold')
axes8[0,1].set_title('Error Metrics Comparison', fontsize=12, fontweight='bold')
axes8[0,1].legend(frameon=True, edgecolor='black', fancybox=False)
axes8[0,1].grid(True, alpha=0.3, axis='y', linestyle='--')
axes8[0,1].tick_params(width=1.5, length=5)
for spine in axes8[0,1].spines.values():
    spine.set_linewidth(2.5)

# Add value labels
for bar in bars3:
    height = bar.get_height()
    axes8[0,1].text(bar.get_x() + bar.get_width()/2., height, f'{height:.2f}', 
                    ha='center', va='bottom', fontsize=9, fontweight='bold')
for bar in bars4:
    height = bar.get_height()
    axes8[0,1].text(bar.get_x() + bar.get_width()/2., height, f'{height:.2f}', 
                    ha='center', va='bottom', fontsize=9, fontweight='bold')

# 8.3 Model Performance Gauge (R²)
from matplotlib.patches import Circle, Wedge
r2_test = metrics_df[metrics_df['Metric']=='R²']['Test'].values[0]

# Create a gauge chart
theta = np.linspace(0, np.pi, 100)
x_gauge = np.cos(theta)
y_gauge = np.sin(theta)

axes8[1,0].plot(x_gauge, y_gauge, 'k-', lw=3)
axes8[1,0].fill_between(x_gauge, 0, y_gauge, alpha=0.1)

# Add colored sectors
colors_gauge = ['#FF0000', '#FF6600', '#FFCC00', '#99CC33', '#339900']
thresholds = [0, 0.2, 0.4, 0.6, 0.8, 1.0]
for i in range(len(thresholds)-1):
    theta_start = np.pi * (1 - thresholds[i+1])
    theta_end = np.pi * (1 - thresholds[i])
    wedge = Wedge((0, 0), 1, theta_start*180/np.pi, theta_end*180/np.pi, 
                  facecolor=colors_gauge[i], alpha=0.5, edgecolor='black', linewidth=2)
    axes8[1,0].add_patch(wedge)

# Add pointer
angle = np.pi * (1 - r2_test)
axes8[1,0].arrow(0, 0, np.cos(angle)*0.8, np.sin(angle)*0.8, 
                  head_width=0.1, head_length=0.1, fc='black', ec='black', linewidth=2.5)
axes8[1,0].add_patch(Circle((0, 0), 0.1, color='white', ec='black', linewidth=2.5))

axes8[1,0].set_xlim(-1.2, 1.2)
axes8[1,0].set_ylim(-0.2, 1.2)
axes8[1,0].set_aspect('equal')
axes8[1,0].axis('off')
axes8[1,0].set_title(f'Model Performance Gauge\nR² = {r2_test:.4f}', fontsize=12, fontweight='bold')

# Add text labels for thresholds
for threshold in [0, 0.2, 0.4, 0.6, 0.8, 1.0]:
    angle_label = np.pi * (1 - threshold)
    x_label = np.cos(angle_label) * 1.05
    y_label = np.sin(angle_label) * 1.05
    axes8[1,0].text(x_label, y_label, f'{threshold:.1f}', ha='center', va='center', fontsize=8, fontweight='bold')

# 8.4 Gap Analysis
metrics_gap = ['R²', 'NSE', 'WI']
gap_values = [metrics_df[metrics_df['Metric']==m]['Gap'].values[0] for m in metrics_gap]
colors_gap = ['#2E8B57' if g > 0 else '#DC143C' for g in gap_values]
bars5 = axes8[1,1].bar(metrics_gap, gap_values, color=colors_gap, alpha=0.7, edgecolor='black', linewidth=2)
axes8[1,1].axhline(y=0, color='black', linestyle='-', lw=2)
axes8[1,1].set_ylabel('Gap (Train - Test)', fontsize=11, fontweight='bold')
axes8[1,1].set_title('Train-Test Performance Gap', fontsize=12, fontweight='bold')
axes8[1,1].grid(True, alpha=0.3, axis='y', linestyle='--')
axes8[1,1].tick_params(width=1.5, length=5)
for spine in axes8[1,1].spines.values():
    spine.set_linewidth(2.5)

# Add value labels
for bar in bars5:
    height = bar.get_height()
    axes8[1,1].text(bar.get_x() + bar.get_width()/2., height, f'{height:.4f}', 
                    ha='center', va='bottom' if height > 0 else 'top', fontsize=10, fontweight='bold')

plt.suptitle('Performance Metrics Dashboard', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(f"{enhanced_dir}/8_Performance_Dashboard.png", dpi=500, bbox_inches='tight', facecolor='white')
plt.close()
print("   ✓ Saved: 8_Performance_Dashboard.png")

# ============================================================================
# SUMMARY
# ============================================================================
print("\n" + "="*80)
print("ALL ENHANCED PLOTS CREATED SUCCESSFULLY!")
print("="*80)
print(f"\nPlots saved in: {enhanced_dir}")
print("\nGenerated Plots (White Theme + Thick Borders):")
print("  1. Enhanced_Predictions_vs_Actual.png - Scatter plot with error bounds")
print("  2. Comprehensive_Residual_Analysis.png - 6-panel residual analysis")
print("  3. Feature_Importance_Analysis.png - Top features + cumulative importance")
print("  4. Cross_Validation_Performance.png - CV scores with error bars")
print("  5. Metrics_Radar_Chart.png - Radar chart for performance metrics")
print("  6. Error_Distribution_Analysis.png - Error distribution statistics")
if has_weights:
    print("  7. Estimator_Weights_Analysis.png - AdaBoost weight analysis")
print("  8. Performance_Dashboard.png - Comprehensive metrics dashboard")
print("\n" + "="*80)
print("FEATURES OF THIS VERSION:")
print("  ✓ White theme (clean background)")
print("  ✓ Thick borders (2.5 linewidth on all axes)")
print("  ✓ Black edges on all elements")
print("  ✓ High contrast for publication")
print("  ✓ 500 DPI resolution")
print("="*80)
print("You can now use these high-quality plots for your paper!")
print("="*80)

LOADING SAVED RESULTS
✓ Data loaded successfully!

CREATING ENHANCED VISUALIZATIONS (WHITE THEME + THICK BORDERS)

1. Creating Enhanced Predictions vs Actual Plot...
   ✓ Saved: 1_Enhanced_Predictions_vs_Actual.png

2. Creating Comprehensive Residual Analysis...
   ✓ Saved: 2_Comprehensive_Residual_Analysis.png

3. Creating Feature Importance Visualizations...
   ✓ Saved: 3_Feature_Importance_Analysis.png

4. Creating Cross-Validation Performance Plot...
   ✓ Saved: 4_Cross_Validation_Performance.png

5. Creating Metrics Radar Chart...
   ✓ Saved: 5_Metrics_Radar_Chart.png

6. Creating Error Distribution Analysis...
   ✓ Saved: 6_Error_Distribution_Analysis.png

7. Creating Estimator Weights Analysis...
   ✓ Saved: 7_Estimator_Weights_Analysis.png

8. Creating Performance Metrics Dashboard...
   ✓ Saved: 8_Performance_Dashboard.png

ALL ENHANCED PLOTS CREATED SUCCESSFULLY!

Plots saved in: D:\2026 Work\My Papers\Predicting calcium carbonate yield\Models\AdaBoost/Enhanced_Plots_White_Th

In [39]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy import stats
from scipy.stats import gaussian_kde, probplot
import os
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import warnings
warnings.filterwarnings('ignore')

# ============================================================================
# CONFIG
# ============================================================================
results_dir = r"D:\2026 Work\My Papers\Predicting calcium carbonate yield\Models\AdaBoost"

print("="*80)
print("LOADING SAVED RESULTS")
print("="*80)

train_results = pd.read_csv(f"{results_dir}/Train_Results.csv")
test_results = pd.read_csv(f"{results_dir}/Test_Results.csv")
metrics_df = pd.read_csv(f"{results_dir}/All_Metrics.csv")

print("Data loaded successfully!")

embedded_dir = f"{results_dir}/Embedded_Error_Plots"
os.makedirs(embedded_dir, exist_ok=True)

# ============================================================================
# HIGH CONTRAST STYLE WITH VERY THICK BORDERS AND LARGER TEXT
# ============================================================================
plt.rcParams['figure.dpi'] = 500
plt.rcParams['savefig.dpi'] = 500
plt.rcParams['font.size'] = 16  # Further increased
plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['axes.labelsize'] = 18  # Further increased
plt.rcParams['axes.labelweight'] = 'bold'
plt.rcParams['axes.titlesize'] = 20  # Further increased
plt.rcParams['axes.titleweight'] = 'bold'
plt.rcParams['legend.fontsize'] = 15  # Further increased
plt.rcParams['axes.linewidth'] = 4.0  # Further increased
plt.rcParams['xtick.major.width'] = 3.5  # Further increased
plt.rcParams['ytick.major.width'] = 3.5  # Further increased
plt.rcParams['xtick.labelsize'] = 15  # Further increased
plt.rcParams['ytick.labelsize'] = 15  # Further increased
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['savefig.facecolor'] = 'white'
plt.rcParams['axes.facecolor'] = 'white'

plt.style.use('default')
sns.set_style("white")

# HIGH CONTRAST color palette
colors = {
    'train': '#1B4FA0',      # Deep Blue
    'test': '#D0021B',       # Strong Red
    'perfect': '#0A7D33',    # Dark Green
    'band10': '#F5A623',     # Amber
    'band20': '#4A90D9',     # Sky Blue
    'density': '#5B0E91',    # Deep Purple
    'grid': '#B0B0B0',
    'text_bg': '#FFFFFF',
    'edge': '#000000',
}

CMAP = 'plasma_r'

def style_axes(ax, lw=4.0):
    """Apply consistent high-contrast border/grid to an axis with thicker lines."""
    for spine in ax.spines.values():
        spine.set_linewidth(lw)
        spine.set_color('black')
    ax.set_facecolor('white')
    ax.grid(True, alpha=0.25, linestyle='--', color=colors['grid'])
    ax.tick_params(width=lw*0.8, length=10, labelsize=15)

# ============================================================================
# DATA PREP
# ============================================================================
y_train_true = train_results['TrueValues'].values
y_train_pred = train_results['PredictedValues'].values
y_test_true = test_results['TrueValues'].values
y_test_pred = test_results['PredictedValues'].values

train_residuals = train_results['Error'].values
test_residuals = test_results['Error'].values
train_abs_error = train_results['AbsoluteError'].values
test_abs_error = test_results['AbsoluteError'].values
train_pct_error = train_results['PercentageError'].values
test_pct_error = test_results['PercentageError'].values

min_val = min(y_train_true.min(), y_train_pred.min(), y_test_true.min(), y_test_pred.min())
max_val = max(y_train_true.max(), y_train_pred.max(), y_test_true.max(), y_test_pred.max())
pad = 0.03 * (max_val - min_val)
min_val, max_val = min_val - pad, max_val + pad

r2_test = r2_score(y_test_true, y_test_pred)
rmse_test = np.sqrt(mean_squared_error(y_test_true, y_test_pred))
mae_test = mean_absolute_error(y_test_true, y_test_pred)
mape_test = np.mean(test_pct_error)

r2_train = r2_score(y_train_true, y_train_pred)
rmse_train = np.sqrt(mean_squared_error(y_train_true, y_train_pred))
mae_train = mean_absolute_error(y_train_true, y_train_pred)
mape_train = np.mean(train_pct_error)

np.random.seed(42)
jitter_amount = 0.015 * (max_val - min_val)

def add_perfect_and_bands(ax, min_val, max_val):
    ax.plot([min_val, max_val], [min_val, max_val],
            color='black', lw=4.5, alpha=0.95, label='Perfect Prediction', zorder=5)
    ax.fill_between([min_val, max_val], [min_val*0.9, max_val*0.9], [min_val*1.1, max_val*1.1],
                    alpha=0.15, color=colors['band10'], label='±10%', zorder=1)
    ax.fill_between([min_val, max_val], [min_val*0.8, max_val*0.8], [min_val*1.2, max_val*1.2],
                    alpha=0.10, color=colors['band20'], label='±20%', zorder=0)

def create_scatter_with_embedded_error(y_true, y_pred, abs_error, residuals, 
                                      r2, rmse, mae, mape, dataset_name, color, 
                                      min_val, max_val, save_path):
    """Create a scatter plot with compact embedded error distribution."""
    
    fig, ax = plt.subplots(figsize=(17, 14))  # Larger figure
    style_axes(ax, lw=4.0)
    
    # Add jitter for better visualization
    x_jitter = y_true + np.random.normal(0, jitter_amount, len(y_true))
    y_jitter = y_pred + np.random.normal(0, jitter_amount, len(y_pred))
    
    # Scatter plot colored by absolute error
    scatter = ax.scatter(x_jitter, y_jitter, c=abs_error, cmap=CMAP,
                         s=90, alpha=0.9, edgecolors='black', linewidth=2.0, zorder=4)
    
    # Add perfect prediction line and error bands
    add_perfect_and_bands(ax, min_val, max_val)
    
    # SMALLER embedded histogram positioned to avoid overlap
    inset_width = 0.22  # Smaller
    inset_height = 0.18  # Smaller
    inset_x = 0.73  # Moved further right
    inset_y = 0.05  # Moved further down
    
    ax_inset = fig.add_axes([inset_x, inset_y, inset_width, inset_height])
    style_axes(ax_inset, lw=3.0)
    
    # Histogram with KDE
    ax_inset.hist(abs_error, bins=12, density=True, alpha=0.8,
                  color=color, edgecolor='black', linewidth=1.5)
    kde = gaussian_kde(abs_error)
    x_range = np.linspace(0, abs_error.max()*1.1, 50)
    ax_inset.plot(x_range, kde(x_range), 'k-', lw=2.5)
    
    # Add mean and median lines
    mean_ae, median_ae = np.mean(abs_error), np.median(abs_error)
    ax_inset.axvline(mean_ae, color=color, linestyle='--', lw=2.0, label=f'Mean: {mean_ae:.2f}%')
    ax_inset.axvline(median_ae, color=colors['perfect'], linestyle='--', lw=2.0, label=f'Median: {median_ae:.2f}%')
    
    # Compact statistics inside inset
    stats_text = f'Mean: {mean_ae:.2f}%\nMedian: {median_ae:.2f}%\nStd: {np.std(abs_error):.2f}%'
    ax_inset.text(0.95, 0.95, stats_text,
                  transform=ax_inset.transAxes, fontsize=9, va='top', ha='right',
                  fontweight='bold',
                  bbox=dict(boxstyle='round', facecolor='white', edgecolor='black', 
                           linewidth=2.0, alpha=0.95))
    
    ax_inset.set_xlabel('|Error| (%)', fontsize=11, fontweight='bold')
    ax_inset.set_ylabel('Density', fontsize=11, fontweight='bold')
    ax_inset.set_title(f'{dataset_name} Error', fontsize=12, fontweight='bold')
    ax_inset.set_xlim(0, abs_error.max()*1.1)
    ax_inset.legend(loc='upper left', fontsize=8, framealpha=0.95, edgecolor='black', 
                   fancybox=False, borderpad=0.5)
    ax_inset.tick_params(labelsize=10)
    
    # Statistics box on main plot with larger text
    stats_box = (f'{dataset_name.upper()} SET PERFORMANCE\n' + '='*28 +
                 f'\n  R²     : {r2:.4f}\n  RMSE   : {rmse:.3f} %\n'
                 f'  MAE    : {mae:.3f} %\n  MAPE   : {mape:.2f} %\n' + '='*28 +
                 f'\n  Samples: {len(y_true)}')
    ax.text(0.02, 0.98, stats_box, transform=ax.transAxes, fontsize=14, va='top',
            fontfamily='monospace', fontweight='bold',
            bbox=dict(boxstyle='round', facecolor='white', edgecolor='black', 
                     linewidth=3.5))
    
    ax.set_xlabel('Actual Calcium Carbonate Yield (%)', fontsize=18, fontweight='bold')
    ax.set_ylabel('Predicted Calcium Carbonate Yield (%)', fontsize=18, fontweight='bold')
    ax.set_title(f'Scatter Plot with Embedded Error Distribution ({dataset_name} Set)', 
                pad=30, fontsize=20, fontweight='bold')
    ax.legend(loc='lower right', framealpha=0.95, edgecolor='black', fancybox=False,
             fontsize=15, borderpad=1.2)
    ax.set_aspect('equal', adjustable='box')
    ax.set_xlim(min_val, max_val)
    ax.set_ylim(min_val, max_val)
    
    cbar = plt.colorbar(scatter, ax=ax, pad=0.02, fraction=0.03)
    cbar.set_label('Absolute Error (%)', fontsize=15, fontweight='bold')
    cbar.ax.tick_params(width=3.5, length=10, labelsize=14)
    
    plt.tight_layout()
    plt.savefig(save_path, dpi=500, bbox_inches='tight', facecolor='white')
    plt.close()
    print(f"   Saved: {os.path.basename(save_path)}")

# ============================================================================
# PLOT 1: Training Set Scatter with Embedded Error Distribution
# ============================================================================
print("\n1. Creating Training Set Scatter Plot with Embedded Error Distribution...")

create_scatter_with_embedded_error(
    y_true=y_train_true,
    y_pred=y_train_pred,
    abs_error=train_abs_error,
    residuals=train_residuals,
    r2=r2_train,
    rmse=rmse_train,
    mae=mae_train,
    mape=mape_train,
    dataset_name='Training',
    color=colors['train'],
    min_val=min_val,
    max_val=max_val,
    save_path=f"{embedded_dir}/1_Training_Scatter_With_Embedded_Error.png"
)

# ============================================================================
# PLOT 2: Testing Set Scatter with Embedded Error Distribution
# ============================================================================
print("\n2. Creating Testing Set Scatter Plot with Embedded Error Distribution...")

create_scatter_with_embedded_error(
    y_true=y_test_true,
    y_pred=y_test_pred,
    abs_error=test_abs_error,
    residuals=test_residuals,
    r2=r2_test,
    rmse=rmse_test,
    mae=mae_test,
    mape=mape_test,
    dataset_name='Testing',
    color=colors['test'],
    min_val=min_val,
    max_val=max_val,
    save_path=f"{embedded_dir}/2_Testing_Scatter_With_Embedded_Error.png"
)

# ============================================================================
# PLOT 3: Side-by-Side Comparison with Compact Insets
# ============================================================================
print("\n3. Creating Side-by-Side Comparison of Training and Testing...")

fig3, (ax_train, ax_test) = plt.subplots(1, 2, figsize=(24, 11))

# Training subplot
style_axes(ax_train, lw=4.0)
x_jitter_train = y_train_true + np.random.normal(0, jitter_amount, len(y_train_true))
y_jitter_train = y_train_pred + np.random.normal(0, jitter_amount, len(y_train_pred))
scatter_train = ax_train.scatter(x_jitter_train, y_jitter_train, c=train_abs_error, cmap=CMAP,
                                 s=75, alpha=0.85, edgecolors='black', linewidth=1.8, zorder=4)
add_perfect_and_bands(ax_train, min_val, max_val)

# Smaller inset for training
inset_width = 0.20
inset_height = 0.16
inset_x = 0.72
inset_y = 0.05
ax_inset_train = fig3.add_axes([inset_x, inset_y, inset_width, inset_height])
style_axes(ax_inset_train, lw=2.8)
ax_inset_train.hist(train_abs_error, bins=12, density=True, alpha=0.8,
                    color=colors['train'], edgecolor='black', linewidth=1.2)
kde_train = gaussian_kde(train_abs_error)
x_range_train = np.linspace(0, train_abs_error.max()*1.1, 50)
ax_inset_train.plot(x_range_train, kde_train(x_range_train), 'k-', lw=2.5)
mean_ae_train = np.mean(train_abs_error)
ax_inset_train.axvline(mean_ae_train, color=colors['perfect'], linestyle='--', lw=2.0)
ax_inset_train.text(0.95, 0.95, f'Mean: {mean_ae_train:.2f}%',
                    transform=ax_inset_train.transAxes, fontsize=8, va='top', ha='right', 
                    fontweight='bold',
                    bbox=dict(boxstyle='round', facecolor='white', edgecolor='black', 
                             linewidth=1.8, alpha=0.95))
ax_inset_train.set_xlabel('|Error| (%)', fontsize=10, fontweight='bold')
ax_inset_train.set_ylabel('Density', fontsize=10, fontweight='bold')
ax_inset_train.set_xlim(0, train_abs_error.max()*1.1)
ax_inset_train.tick_params(labelsize=9)

# Stats for training
stats_box_train = (f'TRAINING SET\n' + '='*18 +
                   f'\nR²: {r2_train:.4f}\nRMSE: {rmse_train:.3f}%\n'
                   f'MAE: {mae_train:.3f}%\nMAPE: {mape_train:.2f}%')
ax_train.text(0.02, 0.98, stats_box_train, transform=ax_train.transAxes, 
              fontsize=13, va='top', fontfamily='monospace', fontweight='bold',
              bbox=dict(boxstyle='round', facecolor='white', edgecolor='black', 
                       linewidth=3.0))
ax_train.set_xlabel('Actual Yield (%)', fontsize=17, fontweight='bold')
ax_train.set_ylabel('Predicted Yield (%)', fontsize=17, fontweight='bold')
ax_train.set_title('Training Set', pad=20, fontsize=18, fontweight='bold')
ax_train.legend(loc='lower right', framealpha=0.95, edgecolor='black', 
               fontsize=13, borderpad=1.0)
ax_train.set_aspect('equal', adjustable='box')
ax_train.set_xlim(min_val, max_val)
ax_train.set_ylim(min_val, max_val)

# Testing subplot
style_axes(ax_test, lw=4.0)
x_jitter_test = y_test_true + np.random.normal(0, jitter_amount, len(y_test_true))
y_jitter_test = y_test_pred + np.random.normal(0, jitter_amount, len(y_test_pred))
scatter_test = ax_test.scatter(x_jitter_test, y_jitter_test, c=test_abs_error, cmap=CMAP,
                               s=75, alpha=0.85, edgecolors='black', linewidth=1.8, zorder=4)
add_perfect_and_bands(ax_test, min_val, max_val)

# Smaller inset for testing
ax_inset_test = fig3.add_axes([inset_x+0.5, inset_y, inset_width, inset_height])
style_axes(ax_inset_test, lw=2.8)
ax_inset_test.hist(test_abs_error, bins=12, density=True, alpha=0.8,
                   color=colors['test'], edgecolor='black', linewidth=1.2)
kde_test = gaussian_kde(test_abs_error)
x_range_test = np.linspace(0, test_abs_error.max()*1.1, 50)
ax_inset_test.plot(x_range_test, kde_test(x_range_test), 'k-', lw=2.5)
mean_ae_test = np.mean(test_abs_error)
ax_inset_test.axvline(mean_ae_test, color=colors['perfect'], linestyle='--', lw=2.0)
ax_inset_test.text(0.95, 0.95, f'Mean: {mean_ae_test:.2f}%',
                   transform=ax_inset_test.transAxes, fontsize=8, va='top', ha='right', 
                   fontweight='bold',
                   bbox=dict(boxstyle='round', facecolor='white', edgecolor='black', 
                            linewidth=1.8, alpha=0.95))
ax_inset_test.set_xlabel('|Error| (%)', fontsize=10, fontweight='bold')
ax_inset_test.set_ylabel('Density', fontsize=10, fontweight='bold')
ax_inset_test.set_xlim(0, test_abs_error.max()*1.1)
ax_inset_test.tick_params(labelsize=9)

# Stats for testing
stats_box_test = (f'TESTING SET\n' + '='*18 +
                  f'\nR²: {r2_test:.4f}\nRMSE: {rmse_test:.3f}%\n'
                  f'MAE: {mae_test:.3f}%\nMAPE: {mape_test:.2f}%')
ax_test.text(0.02, 0.98, stats_box_test, transform=ax_test.transAxes, 
             fontsize=13, va='top', fontfamily='monospace', fontweight='bold',
             bbox=dict(boxstyle='round', facecolor='white', edgecolor='black', 
                      linewidth=3.0))
ax_test.set_xlabel('Actual Yield (%)', fontsize=17, fontweight='bold')
ax_test.set_ylabel('Predicted Yield (%)', fontsize=17, fontweight='bold')
ax_test.set_title('Testing Set', pad=20, fontsize=18, fontweight='bold')
ax_test.legend(loc='lower right', framealpha=0.95, edgecolor='black', 
              fontsize=13, borderpad=1.0)
ax_test.set_aspect('equal', adjustable='box')
ax_test.set_xlim(min_val, max_val)
ax_test.set_ylim(min_val, max_val)

# Colorbar
cbar_ax = fig3.add_axes([0.92, 0.15, 0.018, 0.7])
cbar3 = plt.colorbar(scatter_test, cax=cbar_ax)
cbar3.set_label('Absolute Error (%)', fontsize=15, fontweight='bold')
cbar3.ax.tick_params(width=3.5, length=10, labelsize=14)

plt.suptitle('Training vs Testing: Scatter Plots with Embedded Error Distributions', 
             fontsize=22, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(f"{embedded_dir}/3_Training_Testing_Comparison.png", dpi=500, bbox_inches='tight', facecolor='white')
plt.close()
print("   Saved: 3_Training_Testing_Comparison.png")

# ============================================================================
# PLOT 4: Combined Plot with Error Distributions (Top Section)
# ============================================================================
print("\n4. Creating Combined Error Distribution Plot...")

fig4 = plt.figure(figsize=(18, 12))
gs = gridspec.GridSpec(2, 2, height_ratios=[2, 1], hspace=0.3, wspace=0.3)

# Top-left: Training scatter
ax4a = fig4.add_subplot(gs[0, 0])
style_axes(ax4a, lw=4.0)
x_jitter_train4 = y_train_true + np.random.normal(0, jitter_amount, len(y_train_true))
y_jitter_train4 = y_train_pred + np.random.normal(0, jitter_amount, len(y_train_pred))
scatter4a = ax4a.scatter(x_jitter_train4, y_jitter_train4, c=train_abs_error, cmap=CMAP,
                         s=70, alpha=0.85, edgecolors='black', linewidth=1.5, zorder=4)
add_perfect_and_bands(ax4a, min_val, max_val)
ax4a.set_xlabel('Actual Yield (%)', fontsize=16, fontweight='bold')
ax4a.set_ylabel('Predicted Yield (%)', fontsize=16, fontweight='bold')
ax4a.set_title('Training Set: Actual vs Predicted', fontsize=17, fontweight='bold')
ax4a.legend(loc='lower right', framealpha=0.95, edgecolor='black', fontsize=12)
ax4a.set_aspect('equal', adjustable='box')
ax4a.set_xlim(min_val, max_val)
ax4a.set_ylim(min_val, max_val)

# Top-right: Testing scatter
ax4b = fig4.add_subplot(gs[0, 1])
style_axes(ax4b, lw=4.0)
x_jitter_test4 = y_test_true + np.random.normal(0, jitter_amount, len(y_test_true))
y_jitter_test4 = y_test_pred + np.random.normal(0, jitter_amount, len(y_test_pred))
scatter4b = ax4b.scatter(x_jitter_test4, y_jitter_test4, c=test_abs_error, cmap=CMAP,
                         s=70, alpha=0.85, edgecolors='black', linewidth=1.5, zorder=4)
add_perfect_and_bands(ax4b, min_val, max_val)
ax4b.set_xlabel('Actual Yield (%)', fontsize=16, fontweight='bold')
ax4b.set_ylabel('Predicted Yield (%)', fontsize=16, fontweight='bold')
ax4b.set_title('Testing Set: Actual vs Predicted', fontsize=17, fontweight='bold')
ax4b.legend(loc='lower right', framealpha=0.95, edgecolor='black', fontsize=12)
ax4b.set_aspect('equal', adjustable='box')
ax4b.set_xlim(min_val, max_val)
ax4b.set_ylim(min_val, max_val)

# Bottom-left: Training error distribution
ax4c = fig4.add_subplot(gs[1, 0])
style_axes(ax4c, lw=3.5)
ax4c.hist(train_abs_error, bins=15, density=True, alpha=0.8,
          color=colors['train'], edgecolor='black', linewidth=1.5)
kde_train4 = gaussian_kde(train_abs_error)
x_range_train4 = np.linspace(0, train_abs_error.max()*1.1, 50)
ax4c.plot(x_range_train4, kde_train4(x_range_train4), 'k-', lw=3.0)
mean_train4 = np.mean(train_abs_error)
ax4c.axvline(mean_train4, color=colors['perfect'], linestyle='--', lw=2.5, 
             label=f'Mean: {mean_train4:.3f}%')
ax4c.set_xlabel('Absolute Error (%)', fontsize=15, fontweight='bold')
ax4c.set_ylabel('Density', fontsize=15, fontweight='bold')
ax4c.set_title('Training Error Distribution', fontsize=16, fontweight='bold')
ax4c.legend(fontsize=12, framealpha=0.95, edgecolor='black')
ax4c.set_xlim(0, train_abs_error.max()*1.1)

# Bottom-right: Testing error distribution
ax4d = fig4.add_subplot(gs[1, 1])
style_axes(ax4d, lw=3.5)
ax4d.hist(test_abs_error, bins=15, density=True, alpha=0.8,
          color=colors['test'], edgecolor='black', linewidth=1.5)
kde_test4 = gaussian_kde(test_abs_error)
x_range_test4 = np.linspace(0, test_abs_error.max()*1.1, 50)
ax4d.plot(x_range_test4, kde_test4(x_range_test4), 'k-', lw=3.0)
mean_test4 = np.mean(test_abs_error)
ax4d.axvline(mean_test4, color=colors['perfect'], linestyle='--', lw=2.5,
             label=f'Mean: {mean_test4:.3f}%')
ax4d.set_xlabel('Absolute Error (%)', fontsize=15, fontweight='bold')
ax4d.set_ylabel('Density', fontsize=15, fontweight='bold')
ax4d.set_title('Testing Error Distribution', fontsize=16, fontweight='bold')
ax4d.legend(fontsize=12, framealpha=0.95, edgecolor='black')
ax4d.set_xlim(0, test_abs_error.max()*1.1)

# Add colorbar for scatter plots
cbar_ax4 = fig4.add_axes([0.92, 0.45, 0.018, 0.35])
cbar4 = plt.colorbar(scatter4a, cax=cbar_ax4)
cbar4.set_label('|Error| (%)', fontsize=14, fontweight='bold')
cbar4.ax.tick_params(width=3.0, length=8, labelsize=13)

plt.suptitle('Comprehensive Model Performance Visualization', 
             fontsize=22, fontweight='bold', y=0.98)
plt.tight_layout()
plt.savefig(f"{embedded_dir}/4_Comprehensive_Performance_Plot.png", dpi=500, bbox_inches='tight', facecolor='white')
plt.close()
print("   Saved: 4_Comprehensive_Performance_Plot.png")

# ============================================================================
# PLOT 5: Residuals Analysis with Error Distribution
# ============================================================================
print("\n5. Creating Residuals Analysis Plot...")

fig5 = plt.figure(figsize=(18, 12))
gs5 = gridspec.GridSpec(2, 2, height_ratios=[2, 1], hspace=0.3, wspace=0.3)

# Top-left: Training residuals vs predicted
ax5a = fig5.add_subplot(gs5[0, 0])
style_axes(ax5a, lw=3.5)
ax5a.scatter(y_train_pred, train_residuals, alpha=0.7, s=60, 
             c=colors['train'], edgecolors='black', linewidth=1.2)
ax5a.axhline(0, color='black', lw=3.0, linestyle='-', alpha=0.8)
ax5a.axhline(np.mean(train_residuals), color=colors['perfect'], lw=2.5, 
             linestyle='--', label=f'Mean: {np.mean(train_residuals):.3f}%')
ax5a.set_xlabel('Predicted Yield (%)', fontsize=15, fontweight='bold')
ax5a.set_ylabel('Residuals (%)', fontsize=15, fontweight='bold')
ax5a.set_title('Training: Residuals vs Predicted', fontsize=16, fontweight='bold')
ax5a.legend(fontsize=12, framealpha=0.95, edgecolor='black')

# Top-right: Testing residuals vs predicted
ax5b = fig5.add_subplot(gs5[0, 1])
style_axes(ax5b, lw=3.5)
ax5b.scatter(y_test_pred, test_residuals, alpha=0.7, s=60,
             c=colors['test'], edgecolors='black', linewidth=1.2)
ax5b.axhline(0, color='black', lw=3.0, linestyle='-', alpha=0.8)
ax5b.axhline(np.mean(test_residuals), color=colors['perfect'], lw=2.5,
             linestyle='--', label=f'Mean: {np.mean(test_residuals):.3f}%')
ax5b.set_xlabel('Predicted Yield (%)', fontsize=15, fontweight='bold')
ax5b.set_ylabel('Residuals (%)', fontsize=15, fontweight='bold')
ax5b.set_title('Testing: Residuals vs Predicted', fontsize=16, fontweight='bold')
ax5b.legend(fontsize=12, framealpha=0.95, edgecolor='black')

# Bottom-left: Training residual distribution
ax5c = fig5.add_subplot(gs5[1, 0])
style_axes(ax5c, lw=3.5)
ax5c.hist(train_residuals, bins=15, density=True, alpha=0.8,
          color=colors['train'], edgecolor='black', linewidth=1.5)
kde_train5 = gaussian_kde(train_residuals)
x_range_train5 = np.linspace(train_residuals.min(), train_residuals.max(), 50)
ax5c.plot(x_range_train5, kde_train5(x_range_train5), 'k-', lw=3.0)
ax5c.axvline(0, color='black', lw=2.5, linestyle='-', alpha=0.8)
ax5c.axvline(np.mean(train_residuals), color=colors['perfect'], lw=2.0, 
             linestyle='--', label=f'Mean: {np.mean(train_residuals):.3f}%')
ax5c.set_xlabel('Residuals (%)', fontsize=15, fontweight='bold')
ax5c.set_ylabel('Density', fontsize=15, fontweight='bold')
ax5c.set_title('Training Residual Distribution', fontsize=16, fontweight='bold')
ax5c.legend(fontsize=12, framealpha=0.95, edgecolor='black')

# Bottom-right: Testing residual distribution
ax5d = fig5.add_subplot(gs5[1, 1])
style_axes(ax5d, lw=3.5)
ax5d.hist(test_residuals, bins=15, density=True, alpha=0.8,
          color=colors['test'], edgecolor='black', linewidth=1.5)
kde_test5 = gaussian_kde(test_residuals)
x_range_test5 = np.linspace(test_residuals.min(), test_residuals.max(), 50)
ax5d.plot(x_range_test5, kde_test5(x_range_test5), 'k-', lw=3.0)
ax5d.axvline(0, color='black', lw=2.5, linestyle='-', alpha=0.8)
ax5d.axvline(np.mean(test_residuals), color=colors['perfect'], lw=2.0,
             linestyle='--', label=f'Mean: {np.mean(test_residuals):.3f}%')
ax5d.set_xlabel('Residuals (%)', fontsize=15, fontweight='bold')
ax5d.set_ylabel('Density', fontsize=15, fontweight='bold')
ax5d.set_title('Testing Residual Distribution', fontsize=16, fontweight='bold')
ax5d.legend(fontsize=12, framealpha=0.95, edgecolor='black')

plt.suptitle('Residuals Analysis: Training vs Testing', 
             fontsize=22, fontweight='bold', y=0.98)
plt.tight_layout()
plt.savefig(f"{embedded_dir}/5_Residuals_Analysis.png", dpi=500, bbox_inches='tight', facecolor='white')
plt.close()
print("   Saved: 5_Residuals_Analysis.png")

# ============================================================================
# SUMMARY
# ============================================================================
print("\n" + "="*80)
print("ALL PLOTS CREATED SUCCESSFULLY!")
print("="*80)
print(f"\nPlots saved in: {embedded_dir}")
print("\nGenerated Plots:")
print("  1. Training_Scatter_With_Embedded_Error.png     - Training set with compact error inset")
print("  2. Testing_Scatter_With_Embedded_Error.png      - Testing set with compact error inset")
print("  3. Training_Testing_Comparison.png              - Side-by-side comparison with compact insets")
print("  4. Comprehensive_Performance_Plot.png           - 4-panel comprehensive visualization")
print("  5. Residuals_Analysis.png                       - Residuals analysis with distributions")
print("\nEnhancements made:")
print("  ✓ SMALLER error plots (0.22 x 0.18) for proper placement")
print("  ✓ No merging/overlapping with other elements")
print("  ✓ Further increased text sizes throughout")
print("  ✓ Thicker borders (4.0) for better visibility")
print("  ✓ Added comprehensive 4-panel plot")
print("  ✓ Added residuals analysis plot")
print("  ✓ Optimized inset positioning (x=0.73, y=0.05)")
print("="*80)

LOADING SAVED RESULTS
Data loaded successfully!

1. Creating Training Set Scatter Plot with Embedded Error Distribution...
   Saved: 1_Training_Scatter_With_Embedded_Error.png

2. Creating Testing Set Scatter Plot with Embedded Error Distribution...
   Saved: 2_Testing_Scatter_With_Embedded_Error.png

3. Creating Side-by-Side Comparison of Training and Testing...
   Saved: 3_Training_Testing_Comparison.png

4. Creating Combined Error Distribution Plot...
   Saved: 4_Comprehensive_Performance_Plot.png

5. Creating Residuals Analysis Plot...
   Saved: 5_Residuals_Analysis.png

ALL PLOTS CREATED SUCCESSFULLY!

Plots saved in: D:\2026 Work\My Papers\Predicting calcium carbonate yield\Models\AdaBoost/Embedded_Error_Plots

Generated Plots:
  1. Training_Scatter_With_Embedded_Error.png     - Training set with compact error inset
  2. Testing_Scatter_With_Embedded_Error.png      - Testing set with compact error inset
  3. Training_Testing_Comparison.png              - Side-by-side comparison w

In [57]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy import stats
from scipy.stats import gaussian_kde, probplot
import os
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import warnings
warnings.filterwarnings('ignore')

# ============================================================================
# CONFIG
# ============================================================================
results_dir = r"D:\2026 Work\My Papers\Predicting calcium carbonate yield\Models\AdaBoost"

print("="*80)
print("LOADING SAVED RESULTS")
print("="*80)

train_results = pd.read_csv(f"{results_dir}/Train_Results.csv")
test_results = pd.read_csv(f"{results_dir}/Test_Results.csv")
metrics_df = pd.read_csv(f"{results_dir}/All_Metrics.csv")

print("Data loaded successfully!")

embedded_dir = f"{results_dir}/Embedded_Error_Plots"
os.makedirs(embedded_dir, exist_ok=True)

# ============================================================================
# HIGH CONTRAST STYLE WITH VERY THICK BORDERS AND LARGER TEXT
# ============================================================================
plt.rcParams['figure.dpi'] = 500
plt.rcParams['savefig.dpi'] = 500
plt.rcParams['font.size'] = 16
plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['axes.labelsize'] = 18
plt.rcParams['axes.labelweight'] = 'bold'
plt.rcParams['axes.titlesize'] = 20
plt.rcParams['axes.titleweight'] = 'bold'
plt.rcParams['legend.fontsize'] = 15
plt.rcParams['axes.linewidth'] = 4.0
plt.rcParams['xtick.major.width'] = 3.5
plt.rcParams['ytick.major.width'] = 3.5
plt.rcParams['xtick.labelsize'] = 15
plt.rcParams['ytick.labelsize'] = 15
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['savefig.facecolor'] = 'white'
plt.rcParams['axes.facecolor'] = 'white'

plt.style.use('default')
sns.set_style("white")

# HIGH CONTRAST color palette
colors = {
    'train': '#1B4FA0',      # Deep Blue
    'test': '#D0021B',       # Strong Red
    'perfect': '#0A7D33',    # Dark Green
    'band10': '#F5A623',     # Amber
    'band20': '#4A90D9',     # Sky Blue
    'density': '#5B0E91',    # Deep Purple
    'grid': '#B0B0B0',
    'text_bg': '#FFFFFF',
    'edge': '#000000',
}

CMAP = 'plasma_r'

def style_axes(ax, lw=4.0):
    """Apply consistent high-contrast border/grid to an axis with thicker lines."""
    for spine in ax.spines.values():
        spine.set_linewidth(lw)
        spine.set_color('black')
    ax.set_facecolor('white')
    ax.grid(True, alpha=0.25, linestyle='--', color=colors['grid'])
    ax.tick_params(width=lw*0.8, length=10, labelsize=15)

# ============================================================================
# DATA PREP
# ============================================================================
y_train_true = train_results['TrueValues'].values
y_train_pred = train_results['PredictedValues'].values
y_test_true = test_results['TrueValues'].values
y_test_pred = test_results['PredictedValues'].values

train_residuals = train_results['Error'].values
test_residuals = test_results['Error'].values
train_abs_error = train_results['AbsoluteError'].values
test_abs_error = test_results['AbsoluteError'].values
train_pct_error = train_results['PercentageError'].values
test_pct_error = test_results['PercentageError'].values

min_val = min(y_train_true.min(), y_train_pred.min(), y_test_true.min(), y_test_pred.min())
max_val = max(y_train_true.max(), y_train_pred.max(), y_test_true.max(), y_test_pred.max())
pad = 0.03 * (max_val - min_val)
min_val, max_val = min_val - pad, max_val + pad

r2_test = r2_score(y_test_true, y_test_pred)
rmse_test = np.sqrt(mean_squared_error(y_test_true, y_test_pred))
mae_test = mean_absolute_error(y_test_true, y_test_pred)
mape_test = np.mean(test_pct_error)

r2_train = r2_score(y_train_true, y_train_pred)
rmse_train = np.sqrt(mean_squared_error(y_train_true, y_train_pred))
mae_train = mean_absolute_error(y_train_true, y_train_pred)
mape_train = np.mean(train_pct_error)

np.random.seed(42)
jitter_amount = 0.015 * (max_val - min_val)

def add_perfect_and_bands(ax, min_val, max_val):
    ax.plot([min_val, max_val], [min_val, max_val],
            color='black', lw=4.5, alpha=0.95, label='Perfect Prediction', zorder=5)
    ax.fill_between([min_val, max_val], [min_val*0.9, max_val*0.9], [min_val*1.1, max_val*1.1],
                    alpha=0.15, color=colors['band10'], label='±10%', zorder=1)
    ax.fill_between([min_val, max_val], [min_val*0.8, max_val*0.8], [min_val*1.2, max_val*1.2],
                    alpha=0.10, color=colors['band20'], label='±20%', zorder=0)

def create_scatter_with_embedded_error(y_true, y_pred, abs_error, residuals, 
                                      r2, rmse, mae, mape, dataset_name, color, 
                                      min_val, max_val, save_path):
    """Create a scatter plot with larger embedded error distribution positioned away from data points."""
    
    fig, ax = plt.subplots(figsize=(17, 14))
    style_axes(ax, lw=4.0)
    
    # Add jitter for better visualization
    x_jitter = y_true + np.random.normal(0, jitter_amount, len(y_true))
    y_jitter = y_pred + np.random.normal(0, jitter_amount, len(y_pred))
    
    # Scatter plot colored by absolute error
    scatter = ax.scatter(x_jitter, y_jitter, c=abs_error, cmap=CMAP,
                         s=90, alpha=0.9, edgecolors='black', linewidth=2.0, zorder=4)
    
    # Add perfect prediction line and error bands
    add_perfect_and_bands(ax, min_val, max_val)
    
    # Calculate position with proper gap - bottom-right area with larger size
    x_range = max_val - min_val
    y_range = max_val - min_val
    
    # Position inset in bottom-right area with even larger size
    gap_percent_x = 0.50  # Adjusted for larger inset
    gap_percent_y = 0.06  # Adjusted for larger inset
    
    inset_x_pos = min_val + (gap_percent_x * x_range)
    inset_y_pos = min_val + (gap_percent_y * y_range)
    inset_width = 0.26 * x_range  # Increased from 0.22
    inset_height = 0.21 * y_range  # Increased from 0.18
    
    # Create inset axes using data coordinates
    ax_inset = ax.inset_axes([inset_x_pos, inset_y_pos, inset_width, inset_height], 
                             transform=ax.transData)
    style_axes(ax_inset, lw=3.0)
    
    # Histogram with KDE
    ax_inset.hist(abs_error, bins=12, density=True, alpha=0.8,
                  color=color, edgecolor='black', linewidth=1.5)
    kde = gaussian_kde(abs_error)
    x_range_plot = np.linspace(0, abs_error.max()*1.1, 50)
    ax_inset.plot(x_range_plot, kde(x_range_plot), 'k-', lw=2.5)
    
    # Add mean and median lines
    mean_ae, median_ae = np.mean(abs_error), np.median(abs_error)
    ax_inset.axvline(mean_ae, color=color, linestyle='--', lw=2.0, label=f'Mean: {mean_ae:.2f}%')
    ax_inset.axvline(median_ae, color=colors['perfect'], linestyle='--', lw=2.0, label=f'Median: {median_ae:.2f}%')
    
    # Compact statistics inside inset
    stats_text = f'Mean: {mean_ae:.2f}%\nMedian: {median_ae:.2f}%\nStd: {np.std(abs_error):.2f}%'
    ax_inset.text(0.95, 0.95, stats_text,
                  transform=ax_inset.transAxes, fontsize=11, va='top', ha='right',
                  fontweight='bold',
                  bbox=dict(boxstyle='round', facecolor='white', edgecolor='black', 
                           linewidth=2.0, alpha=0.95))
    
    ax_inset.set_xlabel('|Error| (%)', fontsize=13, fontweight='bold')
    ax_inset.set_ylabel('Density', fontsize=13, fontweight='bold')
    ax_inset.set_title(f'{dataset_name} Error', fontsize=14, fontweight='bold')
    ax_inset.set_xlim(0, abs_error.max()*1.1)
    ax_inset.legend(loc='upper left', fontsize=10, framealpha=0.95, edgecolor='black', 
                   fancybox=False, borderpad=0.5)
    ax_inset.tick_params(labelsize=12)
    
    # Statistics box on main plot with larger text
    stats_box = (f'{dataset_name.upper()} SET PERFORMANCE\n' + '='*28 +
                 f'\n  R²     : {r2:.4f}\n  RMSE   : {rmse:.3f} %\n'
                 f'  MAE    : {mae:.3f} %\n  MAPE   : {mape:.2f} %\n' + '='*28 +
                 f'\n  Samples: {len(y_true)}')
    ax.text(0.02, 0.98, stats_box, transform=ax.transAxes, fontsize=14, va='top',
            fontfamily='monospace', fontweight='bold',
            bbox=dict(boxstyle='round', facecolor='white', edgecolor='black', 
                     linewidth=3.5))
    
    ax.set_xlabel('Actual Calcium Carbonate Yield (%)', fontsize=18, fontweight='bold')
    ax.set_ylabel('Predicted Calcium Carbonate Yield (%)', fontsize=18, fontweight='bold')
    ax.set_title(f'Scatter Plot with Embedded Error Distribution ({dataset_name} Set)', 
                pad=30, fontsize=20, fontweight='bold')
    ax.legend(loc='lower right', framealpha=0.95, edgecolor='black', fancybox=False,
             fontsize=15, borderpad=1.2)
    ax.set_aspect('equal', adjustable='box')
    ax.set_xlim(min_val, max_val)
    ax.set_ylim(min_val, max_val)
    
    cbar = plt.colorbar(scatter, ax=ax, pad=0.02, fraction=0.03)
    cbar.set_label('Absolute Error (%)', fontsize=15, fontweight='bold')
    cbar.ax.tick_params(width=3.5, length=10, labelsize=14)
    
    plt.tight_layout()
    plt.savefig(save_path, dpi=500, bbox_inches='tight', facecolor='white')
    plt.close()
    print(f"   Saved: {os.path.basename(save_path)}")

# ============================================================================
# PLOT 1: Training Set Scatter with Embedded Error Distribution
# ============================================================================
print("\n1. Creating Training Set Scatter Plot with Embedded Error Distribution...")

create_scatter_with_embedded_error(
    y_true=y_train_true,
    y_pred=y_train_pred,
    abs_error=train_abs_error,
    residuals=train_residuals,
    r2=r2_train,
    rmse=rmse_train,
    mae=mae_train,
    mape=mape_train,
    dataset_name='Training',
    color=colors['train'],
    min_val=min_val,
    max_val=max_val,
    save_path=f"{embedded_dir}/1_Training_Scatter_With_Embedded_Error.png"
)

# ============================================================================
# PLOT 2: Testing Set Scatter with Embedded Error Distribution
# ============================================================================
print("\n2. Creating Testing Set Scatter Plot with Embedded Error Distribution...")

create_scatter_with_embedded_error(
    y_true=y_test_true,
    y_pred=y_test_pred,
    abs_error=test_abs_error,
    residuals=test_residuals,
    r2=r2_test,
    rmse=rmse_test,
    mae=mae_test,
    mape=mape_test,
    dataset_name='Testing',
    color=colors['test'],
    min_val=min_val,
    max_val=max_val,
    save_path=f"{embedded_dir}/2_Testing_Scatter_With_Embedded_Error.png"
)

# ============================================================================
# PLOT 3: Side-by-Side Comparison with Larger Insets
# ============================================================================
print("\n3. Creating Side-by-Side Comparison of Training and Testing...")

fig3, (ax_train, ax_test) = plt.subplots(1, 2, figsize=(24, 11))

# Training subplot
style_axes(ax_train, lw=4.0)
x_jitter_train = y_train_true + np.random.normal(0, jitter_amount, len(y_train_true))
y_jitter_train = y_train_pred + np.random.normal(0, jitter_amount, len(y_train_pred))
scatter_train = ax_train.scatter(x_jitter_train, y_jitter_train, c=train_abs_error, cmap=CMAP,
                                 s=75, alpha=0.85, edgecolors='black', linewidth=1.8, zorder=4)
add_perfect_and_bands(ax_train, min_val, max_val)

# Position inset in bottom-right area with larger size
gap_percent_x = 0.50
gap_percent_y = 0.06
x_range = max_val - min_val
y_range = max_val - min_val
inset_x_pos = min_val + (gap_percent_x * x_range)
inset_y_pos = min_val + (gap_percent_y * y_range)
inset_width = 0.23 * x_range  # Increased from 0.20
inset_height = 0.18 * y_range  # Increased from 0.16

ax_inset_train = ax_train.inset_axes([inset_x_pos, inset_y_pos, inset_width, inset_height], 
                                      transform=ax_train.transData)
style_axes(ax_inset_train, lw=2.8)
ax_inset_train.hist(train_abs_error, bins=12, density=True, alpha=0.8,
                    color=colors['train'], edgecolor='black', linewidth=1.2)
kde_train = gaussian_kde(train_abs_error)
x_range_train = np.linspace(0, train_abs_error.max()*1.1, 50)
ax_inset_train.plot(x_range_train, kde_train(x_range_train), 'k-', lw=2.5)
mean_ae_train = np.mean(train_abs_error)
ax_inset_train.axvline(mean_ae_train, color=colors['perfect'], linestyle='--', lw=2.0)
ax_inset_train.text(0.95, 0.95, f'Mean: {mean_ae_train:.2f}%',
                    transform=ax_inset_train.transAxes, fontsize=10, va='top', ha='right', 
                    fontweight='bold',
                    bbox=dict(boxstyle='round', facecolor='white', edgecolor='black', 
                             linewidth=1.8, alpha=0.95))
ax_inset_train.set_xlabel('|Error| (%)', fontsize=12, fontweight='bold')
ax_inset_train.set_ylabel('Density', fontsize=12, fontweight='bold')
ax_inset_train.set_xlim(0, train_abs_error.max()*1.1)
ax_inset_train.tick_params(labelsize=11)

# Stats for training
stats_box_train = (f'TRAINING SET\n' + '='*18 +
                   f'\nR²: {r2_train:.4f}\nRMSE: {rmse_train:.3f}%\n'
                   f'MAE: {mae_train:.3f}%\nMAPE: {mape_train:.2f}%')
ax_train.text(0.02, 0.98, stats_box_train, transform=ax_train.transAxes, 
              fontsize=13, va='top', fontfamily='monospace', fontweight='bold',
              bbox=dict(boxstyle='round', facecolor='white', edgecolor='black', 
                       linewidth=3.0))
ax_train.set_xlabel('Actual Yield (%)', fontsize=17, fontweight='bold')
ax_train.set_ylabel('Predicted Yield (%)', fontsize=17, fontweight='bold')
ax_train.set_title('Training Set', pad=20, fontsize=18, fontweight='bold')
ax_train.legend(loc='lower right', framealpha=0.95, edgecolor='black', 
               fontsize=13, borderpad=1.0)
ax_train.set_aspect('equal', adjustable='box')
ax_train.set_xlim(min_val, max_val)
ax_train.set_ylim(min_val, max_val)

# Testing subplot
style_axes(ax_test, lw=4.0)
x_jitter_test = y_test_true + np.random.normal(0, jitter_amount, len(y_test_true))
y_jitter_test = y_test_pred + np.random.normal(0, jitter_amount, len(y_test_pred))
scatter_test = ax_test.scatter(x_jitter_test, y_jitter_test, c=test_abs_error, cmap=CMAP,
                               s=75, alpha=0.85, edgecolors='black', linewidth=1.8, zorder=4)
add_perfect_and_bands(ax_test, min_val, max_val)

# Position inset in bottom-right area for testing
ax_inset_test = ax_test.inset_axes([inset_x_pos, inset_y_pos, inset_width, inset_height], 
                                    transform=ax_test.transData)
style_axes(ax_inset_test, lw=2.8)
ax_inset_test.hist(test_abs_error, bins=12, density=True, alpha=0.8,
                   color=colors['test'], edgecolor='black', linewidth=1.2)
kde_test = gaussian_kde(test_abs_error)
x_range_test = np.linspace(0, test_abs_error.max()*1.1, 50)
ax_inset_test.plot(x_range_test, kde_test(x_range_test), 'k-', lw=2.5)
mean_ae_test = np.mean(test_abs_error)
ax_inset_test.axvline(mean_ae_test, color=colors['perfect'], linestyle='--', lw=2.0)
ax_inset_test.text(0.95, 0.95, f'Mean: {mean_ae_test:.2f}%',
                   transform=ax_inset_test.transAxes, fontsize=10, va='top', ha='right', 
                   fontweight='bold',
                   bbox=dict(boxstyle='round', facecolor='white', edgecolor='black', 
                            linewidth=1.8, alpha=0.95))
ax_inset_test.set_xlabel('|Error| (%)', fontsize=12, fontweight='bold')
ax_inset_test.set_ylabel('Density', fontsize=12, fontweight='bold')
ax_inset_test.set_xlim(0, test_abs_error.max()*1.1)
ax_inset_test.tick_params(labelsize=11)

# Stats for testing
stats_box_test = (f'TESTING SET\n' + '='*18 +
                  f'\nR²: {r2_test:.4f}\nRMSE: {rmse_test:.3f}%\n'
                  f'MAE: {mae_test:.3f}%\nMAPE: {mape_test:.2f}%')
ax_test.text(0.02, 0.98, stats_box_test, transform=ax_test.transAxes, 
             fontsize=13, va='top', fontfamily='monospace', fontweight='bold',
             bbox=dict(boxstyle='round', facecolor='white', edgecolor='black', 
                      linewidth=3.0))
ax_test.set_xlabel('Actual Yield (%)', fontsize=17, fontweight='bold')
ax_test.set_ylabel('Predicted Yield (%)', fontsize=17, fontweight='bold')
ax_test.set_title('Testing Set', pad=20, fontsize=18, fontweight='bold')
ax_test.legend(loc='lower right', framealpha=0.95, edgecolor='black', 
              fontsize=13, borderpad=1.0)
ax_test.set_aspect('equal', adjustable='box')
ax_test.set_xlim(min_val, max_val)
ax_test.set_ylim(min_val, max_val)

# Colorbar
cbar_ax = fig3.add_axes([0.92, 0.15, 0.018, 0.7])
cbar3 = plt.colorbar(scatter_test, cax=cbar_ax)
cbar3.set_label('Absolute Error (%)', fontsize=15, fontweight='bold')
cbar3.ax.tick_params(width=3.5, length=10, labelsize=14)

plt.suptitle('Training vs Testing: Scatter Plots with Embedded Error Distributions', 
             fontsize=22, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(f"{embedded_dir}/3_Training_Testing_Comparison.png", dpi=500, bbox_inches='tight', facecolor='white')
plt.close()
print("   Saved: 3_Training_Testing_Comparison.png")

# ============================================================================
# PLOT 4: Training Set Scatter with Error Distribution (Clean Version)
# ============================================================================
print("\n4. Creating Clean Training Scatter with Error Distribution...")

fig4, ax4 = plt.subplots(figsize=(17, 14))
style_axes(ax4, lw=4.0)

x_jitter4 = y_train_true + np.random.normal(0, jitter_amount, len(y_train_true))
y_jitter4 = y_train_pred + np.random.normal(0, jitter_amount, len(y_train_pred))
scatter4 = ax4.scatter(x_jitter4, y_jitter4, c=train_abs_error, cmap=CMAP,
                       s=90, alpha=0.9, edgecolors='black', linewidth=2.0, zorder=4)
add_perfect_and_bands(ax4, min_val, max_val)

# Position inset in bottom-right area with larger size
gap_percent_x4 = 0.50
gap_percent_y4 = 0.06
x_range4 = max_val - min_val
y_range4 = max_val - min_val
inset_x_pos4 = min_val + (gap_percent_x4 * x_range4)
inset_y_pos4 = min_val + (gap_percent_y4 * y_range4)
inset_width4 = 0.26 * x_range4  # Increased
inset_height4 = 0.21 * y_range4  # Increased

ax_inset4 = ax4.inset_axes([inset_x_pos4, inset_y_pos4, inset_width4, inset_height4], 
                            transform=ax4.transData)
style_axes(ax_inset4, lw=3.0)

ax_inset4.hist(train_abs_error, bins=12, density=True, alpha=0.8,
               color=colors['train'], edgecolor='black', linewidth=1.5)
kde4 = gaussian_kde(train_abs_error)
x_range4_plot = np.linspace(0, train_abs_error.max()*1.1, 50)
ax_inset4.plot(x_range4_plot, kde4(x_range4_plot), 'k-', lw=2.5)

mean_ae4, median_ae4 = np.mean(train_abs_error), np.median(train_abs_error)
ax_inset4.axvline(mean_ae4, color=colors['train'], linestyle='--', lw=2.0, label=f'Mean: {mean_ae4:.2f}%')
ax_inset4.axvline(median_ae4, color=colors['perfect'], linestyle='--', lw=2.0, label=f'Median: {median_ae4:.2f}%')

stats_text4 = f'Mean: {mean_ae4:.2f}%\nMedian: {median_ae4:.2f}%\nStd: {np.std(train_abs_error):.2f}%'
ax_inset4.text(0.95, 0.95, stats_text4,
               transform=ax_inset4.transAxes, fontsize=11, va='top', ha='right',
               fontweight='bold',
               bbox=dict(boxstyle='round', facecolor='white', edgecolor='black', 
                        linewidth=2.0, alpha=0.95))

ax_inset4.set_xlabel('|Error| (%)', fontsize=13, fontweight='bold')
ax_inset4.set_ylabel('Density', fontsize=13, fontweight='bold')
ax_inset4.set_title('Training Error', fontsize=14, fontweight='bold')
ax_inset4.set_xlim(0, train_abs_error.max()*1.1)
ax_inset4.legend(loc='upper left', fontsize=10, framealpha=0.95, edgecolor='black', 
                fancybox=False, borderpad=0.5)
ax_inset4.tick_params(labelsize=12)

# Statistics box
stats_box4 = (f'TRAINING SET PERFORMANCE\n' + '='*28 +
              f'\n  R²     : {r2_train:.4f}\n  RMSE   : {rmse_train:.3f} %\n'
              f'  MAE    : {mae_train:.3f} %\n  MAPE   : {mape_train:.2f} %\n' + '='*28 +
              f'\n  Samples: {len(y_train_true)}')
ax4.text(0.02, 0.98, stats_box4, transform=ax4.transAxes, fontsize=14, va='top',
         fontfamily='monospace', fontweight='bold',
         bbox=dict(boxstyle='round', facecolor='white', edgecolor='black', 
                  linewidth=3.5))

ax4.set_xlabel('Actual Calcium Carbonate Yield (%)', fontsize=18, fontweight='bold')
ax4.set_ylabel('Predicted Calcium Carbonate Yield (%)', fontsize=18, fontweight='bold')
ax4.set_title('Training Set: Scatter Plot with Error Distribution', 
             pad=30, fontsize=20, fontweight='bold')
ax4.legend(loc='lower right', framealpha=0.95, edgecolor='black', fancybox=False,
          fontsize=15, borderpad=1.2)
ax4.set_aspect('equal', adjustable='box')
ax4.set_xlim(min_val, max_val)
ax4.set_ylim(min_val, max_val)

cbar4 = plt.colorbar(scatter4, ax=ax4, pad=0.02, fraction=0.03)
cbar4.set_label('Absolute Error (%)', fontsize=15, fontweight='bold')
cbar4.ax.tick_params(width=3.5, length=10, labelsize=14)

plt.tight_layout()
plt.savefig(f"{embedded_dir}/4_Training_Clean_Scatter.png", dpi=500, bbox_inches='tight', facecolor='white')
plt.close()
print("   Saved: 4_Training_Clean_Scatter.png")

# ============================================================================
# PLOT 5: Testing Set Scatter with Error Distribution (Clean Version)
# ============================================================================
print("\n5. Creating Clean Testing Scatter with Error Distribution...")

fig5, ax5 = plt.subplots(figsize=(17, 14))
style_axes(ax5, lw=4.0)

x_jitter5 = y_test_true + np.random.normal(0, jitter_amount, len(y_test_true))
y_jitter5 = y_test_pred + np.random.normal(0, jitter_amount, len(y_test_pred))
scatter5 = ax5.scatter(x_jitter5, y_jitter5, c=test_abs_error, cmap=CMAP,
                       s=90, alpha=0.9, edgecolors='black', linewidth=2.0, zorder=4)
add_perfect_and_bands(ax5, min_val, max_val)

# Position inset in bottom-right area with larger size
gap_percent_x5 = 0.50
gap_percent_y5 = 0.06
x_range5 = max_val - min_val
y_range5 = max_val - min_val
inset_x_pos5 = min_val + (gap_percent_x5 * x_range5)
inset_y_pos5 = min_val + (gap_percent_y5 * y_range5)
inset_width5 = 0.26 * x_range5  # Increased
inset_height5 = 0.21 * y_range5  # Increased

ax_inset5 = ax5.inset_axes([inset_x_pos5, inset_y_pos5, inset_width5, inset_height5], 
                            transform=ax5.transData)
style_axes(ax_inset5, lw=3.0)

ax_inset5.hist(test_abs_error, bins=12, density=True, alpha=0.8,
               color=colors['test'], edgecolor='black', linewidth=1.5)
kde5 = gaussian_kde(test_abs_error)
x_range5_plot = np.linspace(0, test_abs_error.max()*1.1, 50)
ax_inset5.plot(x_range5_plot, kde5(x_range5_plot), 'k-', lw=2.5)

mean_ae5, median_ae5 = np.mean(test_abs_error), np.median(test_abs_error)
ax_inset5.axvline(mean_ae5, color=colors['test'], linestyle='--', lw=2.0, label=f'Mean: {mean_ae5:.2f}%')
ax_inset5.axvline(median_ae5, color=colors['perfect'], linestyle='--', lw=2.0, label=f'Median: {median_ae5:.2f}%')

stats_text5 = f'Mean: {mean_ae5:.2f}%\nMedian: {median_ae5:.2f}%\nStd: {np.std(test_abs_error):.2f}%'
ax_inset5.text(0.95, 0.95, stats_text5,
               transform=ax_inset5.transAxes, fontsize=11, va='top', ha='right',
               fontweight='bold',
               bbox=dict(boxstyle='round', facecolor='white', edgecolor='black', 
                        linewidth=2.0, alpha=0.95))

ax_inset5.set_xlabel('|Error| (%)', fontsize=13, fontweight='bold')
ax_inset5.set_ylabel('Density', fontsize=13, fontweight='bold')
ax_inset5.set_title('Testing Error', fontsize=14, fontweight='bold')
ax_inset5.set_xlim(0, test_abs_error.max()*1.1)
ax_inset5.legend(loc='upper left', fontsize=10, framealpha=0.95, edgecolor='black', 
                fancybox=False, borderpad=0.5)
ax_inset5.tick_params(labelsize=12)

# Statistics box
stats_box5 = (f'TESTING SET PERFORMANCE\n' + '='*28 +
              f'\n  R²     : {r2_test:.4f}\n  RMSE   : {rmse_test:.3f} %\n'
              f'  MAE    : {mae_test:.3f} %\n  MAPE   : {mape_test:.2f} %\n' + '='*28 +
              f'\n  Samples: {len(y_test_true)}')
ax5.text(0.02, 0.98, stats_box5, transform=ax5.transAxes, fontsize=14, va='top',
         fontfamily='monospace', fontweight='bold',
         bbox=dict(boxstyle='round', facecolor='white', edgecolor='black', 
                  linewidth=3.5))

ax5.set_xlabel('Actual Calcium Carbonate Yield (%)', fontsize=18, fontweight='bold')
ax5.set_ylabel('Predicted Calcium Carbonate Yield (%)', fontsize=18, fontweight='bold')
ax5.set_title('Testing Set: Scatter Plot with Error Distribution', 
             pad=30, fontsize=20, fontweight='bold')
ax5.legend(loc='lower right', framealpha=0.95, edgecolor='black', fancybox=False,
          fontsize=15, borderpad=1.2)
ax5.set_aspect('equal', adjustable='box')
ax5.set_xlim(min_val, max_val)
ax5.set_ylim(min_val, max_val)

cbar5 = plt.colorbar(scatter5, ax=ax5, pad=0.02, fraction=0.03)
cbar5.set_label('Absolute Error (%)', fontsize=15, fontweight='bold')
cbar5.ax.tick_params(width=3.5, length=10, labelsize=14)

plt.tight_layout()
plt.savefig(f"{embedded_dir}/5_Testing_Clean_Scatter.png", dpi=500, bbox_inches='tight', facecolor='white')
plt.close()
print("   Saved: 5_Testing_Clean_Scatter.png")

# ============================================================================
# PLOT 6: Training and Testing Error Distributions Comparison
# ============================================================================
print("\n6. Creating Error Distribution Comparison Plot...")

fig6, (ax6a, ax6b) = plt.subplots(1, 2, figsize=(20, 9))

# Training error distribution
style_axes(ax6a, lw=3.5)
ax6a.hist(train_abs_error, bins=15, density=True, alpha=0.8,
          color=colors['train'], edgecolor='black', linewidth=1.5)
kde6a = gaussian_kde(train_abs_error)
x_range6a = np.linspace(0, max(train_abs_error.max(), test_abs_error.max())*1.1, 50)
ax6a.plot(x_range6a, kde6a(x_range6a), 'k-', lw=3.0)
mean6a = np.mean(train_abs_error)
ax6a.axvline(mean6a, color=colors['perfect'], linestyle='--', lw=2.5, 
             label=f'Mean: {mean6a:.3f}%')
ax6a.set_xlabel('Absolute Error (%)', fontsize=16, fontweight='bold')
ax6a.set_ylabel('Density', fontsize=16, fontweight='bold')
ax6a.set_title('Training Error Distribution', fontsize=18, fontweight='bold')
ax6a.legend(fontsize=14, framealpha=0.95, edgecolor='black')
ax6a.set_xlim(0, max(train_abs_error.max(), test_abs_error.max())*1.1)

# Testing error distribution
style_axes(ax6b, lw=3.5)
ax6b.hist(test_abs_error, bins=15, density=True, alpha=0.8,
          color=colors['test'], edgecolor='black', linewidth=1.5)
kde6b = gaussian_kde(test_abs_error)
ax6b.plot(x_range6a, kde6b(x_range6a), 'k-', lw=3.0)
mean6b = np.mean(test_abs_error)
ax6b.axvline(mean6b, color=colors['perfect'], linestyle='--', lw=2.5,
             label=f'Mean: {mean6b:.3f}%')
ax6b.set_xlabel('Absolute Error (%)', fontsize=16, fontweight='bold')
ax6b.set_ylabel('Density', fontsize=16, fontweight='bold')
ax6b.set_title('Testing Error Distribution', fontsize=18, fontweight='bold')
ax6b.legend(fontsize=14, framealpha=0.95, edgecolor='black')
ax6b.set_xlim(0, max(train_abs_error.max(), test_abs_error.max())*1.1)

plt.suptitle('Training vs Testing Error Distribution Comparison', 
             fontsize=22, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(f"{embedded_dir}/6_Error_Distribution_Comparison.png", dpi=500, bbox_inches='tight', facecolor='white')
plt.close()
print("   Saved: 6_Error_Distribution_Comparison.png")

# ============================================================================
# SUMMARY
# ============================================================================
print("\n" + "="*80)
print("ALL PLOTS CREATED SUCCESSFULLY!")
print("="*80)
print(f"\nPlots saved in: {embedded_dir}")
print("\nGenerated Plots:")
print("  1. Training_Scatter_With_Embedded_Error.png     - Training set with larger error inset")
print("  2. Testing_Scatter_With_Embedded_Error.png      - Testing set with larger error inset")
print("  3. Training_Testing_Comparison.png              - Side-by-side comparison")
print("  4. Training_Clean_Scatter.png                   - Clean training scatter")
print("  5. Testing_Clean_Scatter.png                    - Clean testing scatter")
print("  6. Error_Distribution_Comparison.png            - Error distributions comparison")
print("\nKey improvements:")
print("  ✓ Error plots size INCREASED FURTHER (26% width, 21% height)")
print("  ✓ Still positioned away from data points and agreement line")
print("  ✓ Adjusted positioning (x=50%, y=6%) for larger size")
print("  ✓ Proper gap from borders maintained")
print("  ✓ No overlapping with any elements")
print("  ✓ Even larger text inside error plots for better readability")
print("  ✓ Legend and labels increased for clarity")
print("="*80)

LOADING SAVED RESULTS
Data loaded successfully!

1. Creating Training Set Scatter Plot with Embedded Error Distribution...
   Saved: 1_Training_Scatter_With_Embedded_Error.png

2. Creating Testing Set Scatter Plot with Embedded Error Distribution...
   Saved: 2_Testing_Scatter_With_Embedded_Error.png

3. Creating Side-by-Side Comparison of Training and Testing...
   Saved: 3_Training_Testing_Comparison.png

4. Creating Clean Training Scatter with Error Distribution...
   Saved: 4_Training_Clean_Scatter.png

5. Creating Clean Testing Scatter with Error Distribution...
   Saved: 5_Testing_Clean_Scatter.png

6. Creating Error Distribution Comparison Plot...
   Saved: 6_Error_Distribution_Comparison.png

ALL PLOTS CREATED SUCCESSFULLY!

Plots saved in: D:\2026 Work\My Papers\Predicting calcium carbonate yield\Models\AdaBoost/Embedded_Error_Plots

Generated Plots:
  1. Training_Scatter_With_Embedded_Error.png     - Training set with larger error inset
  2. Testing_Scatter_With_Embedded_Erro